# Temporal Drift in Visual Embedding Spaces of Multilingual Vision-Language Models

**Authors:**
- **Md Zunayedul Islam** | Universidad Autónoma de Madrid, Madrid, Spain | mdz.islam@estudiante.uam.es
- **Dr. Marcos Escudero-Viñolo** | Universidad Autónoma de Madrid, Madrid, Spain | marcos.escudero@uam.es

---

## Abstract

Contrastive vision-language models encode a shared embedding space implicitly anchored to the temporal distribution of their training data. This notebook presents the first systematic study of *temporal drift* in visual embedding spaces: the phenomenon by which concepts with changing visual appearance produce systematically different image distributions across historical periods. Using 19,334 timestamped photographs from YFCC15M [Thomee et al., 2016] spanning 2004-2014, we probe 18 visual concepts across three semantic tiers with temporally-anchored prompts in six typologically diverse languages and compare three architectures: CLIP [Radford et al., 2021], M-OpenCLIP [Schuhmann et al., 2022; Ilharco et al., 2021], and SigLIP-2 [Tschannen et al., 2025].

---

## Hypotheses

**H1 (Concept-tier drift):** Semantic tier is a significant predictor of visual distributional shift, either directly or in interaction with linguistic resources.

**H2 (Cross-lingual asymmetry):** Temporal text-direction drift is jointly predicted by language training-data volume and morphological typology. High-resource languages with lexical-substitution patterns (Japanese, Russian) produce greater query displacement than low-resource languages (Bengali, Swahili).

**H3 (Trajectory smoothness):** Consecutive centroid displacements across temporal bins are directionally more consistent than a permutation-null baseline, indicating structured evolution rather than random fluctuation.

---

## Metrics Summary

| Metric | Symbol | Description |
|---|---|---|
| Sliced Wasserstein Distance | SWD | Full distributional divergence between embedding clouds [Rabin et al., 2012] |
| Centroid Displacement | $\Delta$ | Cosine distance between L2-normalised mean embeddings across bins |
| Text Direction Drift | $\delta$ | Cosine distance between mean temporally-anchored text embeddings across bins |
| Trajectory Angle | $\theta_{\text{vis}}$ | Angle between consecutive centroid displacement vectors (lower = more directional) |
| MAP@k | MAP@k | Mean Average Precision at cutoff $k$ for neutral-prompt retrieval |

## Statistical Tests

| Hypothesis | Primary Test | Secondary Test |
|---|---|---|
| H1 | LME: tier x log-resource [Pinheiro & Bates, 2000] | Paired permutation test + Cohen's $d$ |
| H2 | Spearman rank correlation [Spearman, 1904] | LME: $\delta \sim \log(\text{resource})$ |
| H3 | Wilcoxon signed-rank test [Wilcoxon, 1945] | 1,000-iteration permutation null |

Significance level: $\alpha = 0.05$ for all tests. All hypotheses are pre-specified before data analysis.

## 1. Environment Setup

In [ ]:
!pip install -q aiohttp nest_asyncio datasets regex python-dotenv
!pip install -q tqdm pandas matplotlib seaborn pingouin
!pip install -q transformers sentencepiece ftfy scikit-learn scikit-image
!pip install -q umap-learn sentence-transformers timm POT
!pip install -q statsmodels opencv-python-headless open-clip-torch

print("All dependencies installed.")

## 2. Imports

In [ ]:
import os
import re
import sys
import glob
import json
import math
import random
import shutil
import warnings
import itertools
import importlib
import dotenv

import aiohttp
import asyncio
import nest_asyncio
from tqdm import tqdm
from tqdm.asyncio import tqdm as atqdm
from IPython.display import display

import pandas as pd
import numpy as np
import ot
import pingouin as pg
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import umap as umap_lib
from scipy.stats import (
    wilcoxon,
    spearmanr,
    ttest_rel,
    kruskal,
    mannwhitneyu
)

import cv2
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms
from torch.utils.data import Dataset, DataLoader

import open_clip
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    CLIPProcessor,
    CLIPModel,
    AutoModel,
    AutoProcessor
)

from huggingface_hub import login

warnings.filterwarnings("ignore")
hf_token = dotenv.read(".env").get("HF_TOKEN")
login(token=hf_token)
print("Libraries imported and HuggingFace login complete.")

## 3. Global Configuration

The `CONFIG` dictionary centralises all experimental hyperparameters.

**Languages and mC4 approximate token counts** [Xue et al., 2021]:

| Language | Code | mC4 tokens (approx.) |
|---|---|---|
| English | `en` | ~400B |
| Russian | `ru` | ~80B |
| Japanese | `ja` | ~40B |
| Arabic | `ar` | ~10B |
| Bengali | `bn` | ~1B |
| Swahili | `sw` | ~0.3B |

Only the relative ordering is used in regression analyses; the absolute counts are order-of-magnitude estimates.

**Vision models:**

| Model | Architecture | Training corpus | Text encoder | Embedding dim |
|---|---|---|---|---|
| CLIP | ViT-B/32 | WebImageText (~400M) | M-CLIP (XLM-RoBERTa) | 512 |
| M-OpenCLIP | ViT-B/32 | LAION-5B | XLM-RoBERTa (joint) | 512 |
| SigLIP-2 | ViT-B/16 | Multilingual (sigmoid loss) | SigLIP-2 native | 768 |

**Concept tiers** are assigned a priori based on expected visual change during 2004-2014.

In [ ]:
WORK_DIR    = ""
META_DIR    = os.path.join(WORK_DIR, "metadata")
EMBED_DIR   = os.path.join(WORK_DIR, "embeddings")
IMAGES_DIR  = os.path.join(WORK_DIR, "images")
RESULTS_DIR = os.path.join(WORK_DIR, "results")

for d in [META_DIR, EMBED_DIR, IMAGES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

device   = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"Device: {device} | {gpu_name}")
print(f"Work dir: {WORK_DIR}")

CONFIG = {
    "languages": ["en", "ru", "ja", "ar", "bn", "sw"],

    "temporal_bins": ["pre2008", "2008_2011", "2012_2014"],
    "k_values"     : [10, 50],
    "n_proj_swd"   : 200,
    "n_permutations": 1000,
    "target_per_bin": 400,

    "models": {
        "clip": {
            "type"         : "hf",
            "name"         : "openai/clip-vit-base-patch32",
            "file_prefix"  : "clip_vitb32",
            "embedding_dim": 512,
        },
        "m_openclip": {
            "type"         : "oc_ml",
            "name"         : "xlm-roberta-base-ViT-B-32",
            "pretrained"   : "laion5b_s13b_b90k",
            "file_prefix"  : "m_openclip_vitb32",
            "embedding_dim": 512,
        },
        "siglip2": {
            "type"         : "siglip2",
            "name"         : "google/siglip2-base-patch16-256",
            "file_prefix"  : "siglip2_ml",
            "embedding_dim": 768,
        },
    },

    "concepts": [
        "mountain", "river", "ocean", "forest", "beach", "moon",
        "hospital", "office", "classroom", "restaurant", "library", "surgery",
        "phone", "protest", "television", "computer", "camera", "car",
    ],
    "concept_tiers": {
        "stable": ["mountain", "river", "ocean", "forest", "beach", "moon"],
        "slow"  : ["hospital", "office", "classroom", "restaurant", "library", "surgery"],
        "fast"  : ["phone", "protest", "television", "computer", "camera", "car"],
    },

    # Approximate token counts from Xue et al. (2021); relative ordering only.
    "resource_level": {
        "en": 400e9,
        "ru": 80e9,
        "ja": 40e9,
        "ar": 10e9,
        "bn": 1e9,
        "sw": 0.3e9,
    },
}

TIER_MAP  = {c: t for t, cs in CONFIG["concept_tiers"].items() for c in cs}
MODEL_DIM = {mk: mv["embedding_dim"] for mk, mv in CONFIG["models"].items()}

print(f"Languages : {CONFIG['languages']}  ({len(CONFIG['languages'])} total)")
print(f"Concepts  : {len(CONFIG['concepts'])} total — "
      f"{len(CONFIG['concept_tiers']['stable'])} stable, "
      f"{len(CONFIG['concept_tiers']['slow'])} slow, "
      f"{len(CONFIG['concept_tiers']['fast'])} fast")
print(f"Models    : {list(CONFIG['models'].keys())}  dims={MODEL_DIM}")

print("Configuration loaded.")

In [ ]:
mpl.rcParams.update({
    "font.family"          : "sans-serif",
    "font.sans-serif"      : ["Helvetica", "DejaVu Sans"],
    "axes.titlesize"       : 12,
    "axes.labelsize"       : 10,
    "xtick.labelsize"      : 9,
    "ytick.labelsize"      : 9,
    "legend.fontsize"      : 9,
    "figure.dpi"           : 100,
    "figure.facecolor"     : "white",
    "axes.spines.top"      : False,
    "axes.spines.right"    : False,
})

## 4. Reproducibility

Global random seed 42 is applied to all stochastic operations: NumPy, PyTorch, Python `random`, and `PYTHONHASHSEED`. All computationally expensive results (embeddings, proxy SWD, sharpness, style SWD, LAB statistics, SN embeddings) are cached to Google Drive so the pipeline can be resumed without recomputation.

In [ ]:
def set_seed(seed: int = 42):
    # Fix all random sources for reproducibility across runs.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)
print("Seed set to 42.")


# Section I: Data Pipeline

The pipeline proceeds in five stages:

1. **Detection** - identify which concepts are missing images on Drive.
2. **Streaming** - pull YFCC15M [Thomee et al., 2016] records using a 2D quota system.
3. **Download** - asynchronously download Flickr images at up to 50 concurrent connections.
4. **Extraction** - encode images with each vision model and cache embeddings.
5. **Verification** - confirm all artifacts are present and L2-normalised.

**Dataset:** YFCC100M accessed via the `mehdidc/yfcc15m` HuggingFace streaming split. Records are matched to concept keywords in the title and description fields. Temporal binning uses the Flickr EXIF upload date.

**Temporal bins:**

| Bin | Period | Rationale |
|---|---|---|
| `pre2008` | Before 2008 | Pre-smartphone-mass-market era |
| `2008_2011` | 2008-2011 | iPhone introduction (2007) and Android ecosystem rise |
| `2012_2014` | 2012-2014 | Post-smartphone convergence; end of YFCC coverage |

The bins are unequal in length due to YFCC's temporal composition (Flickr upload volumes before 2006 are substantially lower). Per-bin quota sampling to 400 images mitigates the resulting image-count imbalance.

## 1.1 Missing Data Detection

In [ ]:
print("Detecting missing data\n")

TARGET = CONFIG["target_per_bin"]
bins   = CONFIG["temporal_bins"]

image_counts = {}
for c in CONFIG["concepts"]:
    image_counts[c] = {}
    for b in bins:
        p = os.path.join(IMAGES_DIR, c, b)
        image_counts[c][b] = (len(glob.glob(os.path.join(p, "*.jpg")))
                               if os.path.isdir(p) else 0)

MISSING_CONCEPTS = [
    c for c in CONFIG["concepts"]
    if any(image_counts[c][b] == 0 for b in bins)
]

print(f"{'CONCEPT':<14} | {'pre2008':>8} {'2008_2011':>10} {'2012_2014':>10} | STATUS")
print("-" * 60)
for c in CONFIG["concepts"]:
    counts = [image_counts[c][b] for b in bins]
    ok     = all(n > 0 for n in counts)
    status = "OK" if ok else "ERROR"
    print(f"  {c:<12} | {counts[0]:>8} {counts[1]:>10} {counts[2]:>10} | {status}")

print()
if not MISSING_CONCEPTS:
    print(" All concepts have images.")
else:
    print(f"  Concepts requiring streaming + download: {MISSING_CONCEPTS}")
    print(f"  Target: {TARGET} images per bin per concept")

## 1.2 Image Streaming

YFCC15M is streamed in a single sequential pass. A **2D quota system** controls acquisition: streaming halts only when all (concept, temporal bin) cells simultaneously reach the target count of 400 images, preventing chronological sampling bias.

In [ ]:
if not MISSING_CONCEPTS:
    print("Streaming skipped. No missing concepts.")
else:
    csv_filtered_path = os.path.join(META_DIR, "yfcc_filtered_metadata.csv")
    TARGET = CONFIG["target_per_bin"]
    bins   = CONFIG["temporal_bins"]

    quota = {}
    for c in MISSING_CONCEPTS:
        quota[c] = {}
        for b in bins:
            have = image_counts[c][b]
            quota[c][b] = max(0, TARGET - have)

    still_needed = {c: b for c, bdict in quota.items()
                    for b, need in bdict.items() if need > 0}
    if not still_needed:
        print("Streaming skipped — quota already satisfied in existing metadata.")
    else:
        concept_regex = {c: re.compile(rf"\b{c}\b", re.IGNORECASE)
                         for c in MISSING_CONCEPTS}
        # Streaming quota: how many MORE rows we want per (concept, bin)
        stream_quota = {c: {b: TARGET for b in bins} for c in MISSING_CONCEPTS}

        # Subtract rows already in filtered CSV
        if os.path.exists(csv_filtered_path):
            df_existing_f = pd.read_csv(csv_filtered_path)
            for c in MISSING_CONCEPTS:
                for b in bins:
                    already = len(df_existing_f[
                        (df_existing_f["matched_concept"] == c) &
                        (df_existing_f["temp_bin"] == b)
                    ])
                    stream_quota[c][b] = max(0, TARGET - already)
        else:
            df_existing_f = None

        records = []
        print(f"Streaming YFCC15M for: {MISSING_CONCEPTS}")
        dataset = load_dataset("mehdidc/yfcc15m", split="train", streaming=True)

        for i, row in enumerate(dataset):
            if i % 500_000 == 0 and i > 0:
                filled  = sum(sum(b.values()) for b in stream_quota.values())
                total_t = sum(sum(bdict.values())
                              for bdict in stream_quota.values())
                remaining = sum(sum(max(0,v) for v in b.values())
                                for b in stream_quota.values())
                print(f"  Scanned {i:,} rows ({remaining} slots remaining)")

            text = (str(row.get("title", "")) + " " +
                    str(row.get("description", ""))).lower()

            for c in MISSING_CONCEPTS:
                if concept_regex[c].search(text):
                    date_taken = str(row.get("datetaken", ""))
                    year_str   = date_taken.split("-")[0] if "-" in date_taken else ""
                    if not year_str.isdigit():
                        break
                    yr = int(year_str)
                    if not (2004 <= yr <= 2014):
                        break
                    t_bin = ("pre2008"   if yr < 2008  else
                             "2008_2011" if yr <= 2011 else
                             "2012_2014")
                    if stream_quota[c][t_bin] > 0:
                        row_copy = dict(row)
                        row_copy.update({"matched_concept": c,
                                         "year": yr, "temp_bin": t_bin})
                        records.append(row_copy)
                        stream_quota[c][t_bin] -= 1
                    break

            if all(v == 0 for bdict in stream_quota.values()
                   for v in bdict.values()):
                print("\nAll quotas filled. Stream stopped.")
                break

        df_new = pd.DataFrame(records)
        print(f"\nNew metadata rows collected: {len(df_new)}")
        if len(df_new):
            print(df_new.groupby(["matched_concept", "temp_bin"])
                        .size().unstack(fill_value=0))

        # Append to existing or create
        if df_existing_f is not None and len(df_new):
            df_combined = pd.concat([df_existing_f, df_new], ignore_index=True)
        elif df_existing_f is not None:
            df_combined = df_existing_f
        else:
            df_combined = df_new

        df_combined.to_csv(csv_filtered_path, index=False)
        print(f"\nSaved: {csv_filtered_path}  ({len(df_combined)} total rows)")

## 1.3 Asynchronous Image Downloader

Images are downloaded from Flickr using an `asyncio`-based pipeline with a semaphore-bounded concurrency of 50 connections. Each file is validated by checking that the downloaded content exceeds 5 KB before being written to disk.

In [ ]:
if not MISSING_CONCEPTS:
    print("Downloading skipped. No missing concepts.")
else:
    nest_asyncio.apply()

    csv_filtered_path = os.path.join(META_DIR, "yfcc_filtered_metadata.csv")
    if not os.path.exists(csv_filtered_path):
        print("No filtered metadata CSV found.")
    else:
        df_all = pd.read_csv(csv_filtered_path)
        df_to_dl = df_all[df_all["matched_concept"].isin(MISSING_CONCEPTS)].copy()
        print(f"Rows to attempt downloading: {len(df_to_dl)}")

        async def fetch_image(session, sem, url, filepath, row_idx):
            if pd.isna(url) or url == "":
                return row_idx, False
            if os.path.exists(filepath):
                return row_idx, True
            try:
                async with sem:
                    headers = {"User-Agent": "Mozilla/5.0"}
                    timeout = aiohttp.ClientTimeout(total=10)
                    async with session.get(url, headers=headers,
                                           timeout=timeout) as resp:
                        if resp.status == 200:
                            content = await resp.read()
                            if len(content) > 5_000:
                                with open(filepath, "wb") as fh:
                                    fh.write(content)
                                return row_idx, True
                return row_idx, False
            except Exception:
                return row_idx, False

        async def download_all(df):
            sem   = asyncio.Semaphore(50)
            tasks = []
            async with aiohttp.ClientSession() as session:
                for idx, row in df.iterrows():
                    try:
                        url = (f"https://farm{int(row['farmid'])}.staticflickr.com/"
                               f"{int(row['serverid'])}/{int(row['photoid'])}"
                               f"_{row['secret']}.jpg")
                    except Exception:
                        url = str(row.get("downloadurl", "")).replace("http:", "https:")

                    concept   = row["matched_concept"]
                    t_bin     = row["temp_bin"]
                    target_dir = os.path.join(IMAGES_DIR, concept, t_bin)
                    os.makedirs(target_dir, exist_ok=True)
                    filepath  = os.path.join(target_dir, f"img_{idx}.jpg")
                    tasks.append(asyncio.create_task(
                        fetch_image(session, sem, url, filepath, idx)))
                return await atqdm.gather(*tasks, desc="Downloading images")

        results     = await download_all(df_to_dl)
        success_map = {idx: ok for idx, ok in results}
        df_to_dl["downloaded"] = df_to_dl.index.map(success_map)

        n_ok = int(df_to_dl["downloaded"].sum())
        print(f"\n Successfully downloaded: {n_ok} / {len(df_to_dl)} images")
        print("\nImages on disk after download:")
        for c in MISSING_CONCEPTS:
            for b in CONFIG["temporal_bins"]:
                p = os.path.join(IMAGES_DIR, c, b)
                n = len(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else 0
                print(f"  {c:<14} | {b:<12} | {n:4d}")

## 1.4 Visual Embedding Extraction

Images are encoded with each of the three vision models in batches of 128. All embeddings are immediately **L2-normalised** to the unit hypersphere $\mathbb{S}^{d-1}$.

In [ ]:
set_seed(42)
BATCH_SIZE = 128


# Yields (preprocessed_tensor, concept, bin, filename) tuples for DataLoader.
class ConceptBinDataset(Dataset):
    def __init__(self, images_dir, concepts, bins, preprocess_fn):
        self.samples    = []
        self.preprocess = preprocess_fn
        for c in concepts:
            for b in bins:
                folder = os.path.join(images_dir, c, b)
                if not os.path.isdir(folder):
                    continue
                for fpath in sorted(glob.glob(os.path.join(folder, "*.jpg"))):
                    self.samples.append((fpath, c, b, os.path.basename(fpath)))
        print(f"  Dataset: {len(self.samples)} images")

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        fpath, concept, t_bin, fname = self.samples[idx]
        try:
            img = Image.open(fpath).convert("RGB")
            return self.preprocess(img), concept, t_bin, fname
        except Exception:
            return None, concept, t_bin, fname


def collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None
    pv, c, t, f = zip(*batch)
    return torch.stack(pv), list(c), list(t), list(f)


def extract_embeddings_for_concepts(model_key, cfg, concepts_to_embed):
    prefix    = cfg["file_prefix"]
    emb_path  = os.path.join(EMBED_DIR, f"{prefix}_embeddings.npy")
    meta_path = os.path.join(EMBED_DIR, f"{prefix}_meta.csv")

    print(f"\n  Loading {model_key} ({cfg['type']}) ...")

    if cfg["type"] == "hf":
        processor = CLIPProcessor.from_pretrained(cfg["name"])
        hf_model  = CLIPModel.from_pretrained(cfg["name"]).to(device)
        hf_model.eval()
        preprocess = lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        def encode_batch(pv):
            out  = hf_model.vision_model(pixel_values=pv.to(device))
            embs = hf_model.visual_projection(out.pooler_output)
            return F.normalize(embs, p=2, dim=-1)

    elif cfg["type"] == "oc_ml":
        oc_model, _, oc_preprocess = open_clip.create_model_and_transforms(
            cfg["name"], pretrained=cfg["pretrained"])
        oc_model = oc_model.to(device).eval()
        preprocess = oc_preprocess
        def encode_batch(pv):
            return F.normalize(oc_model.encode_image(pv.to(device)), p=2, dim=-1)

    elif cfg["type"] == "siglip2":
        sig2_proc  = AutoProcessor.from_pretrained(cfg["name"])
        sig2_model = AutoModel.from_pretrained(cfg["name"]).to(device)
        sig2_model.eval()
        preprocess = lambda img: sig2_proc(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        def encode_batch(pv):
            out  = sig2_model.vision_model(pixel_values=pv.to(device))
            embs = out.pooler_output
            return F.normalize(embs, p=2, dim=-1)

    else:
        raise ValueError(f"Unknown model type: {cfg['type']}")

    ds = ConceptBinDataset(IMAGES_DIR, concepts_to_embed, CONFIG["temporal_bins"], preprocess)
    if len(ds) == 0:
        print(f"  No images found. Skipping extraction.")
        return

    dl = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=2,
                    collate_fn=collate, pin_memory=(device == "cuda"))

    new_embs, new_meta = [], []
    with torch.no_grad():
        for batch in tqdm(dl, desc=f"  Extracting [{model_key}]"):
            if batch is None: continue
            pv, c, t, f = batch
            embs = encode_batch(pv)
            new_embs.append(embs.cpu().numpy())
            for ci, ti, fi in zip(c, t, f):
                new_meta.append({"concept": ci, "temp_bin": ti, "filename": fi})

    new_arr = np.vstack(new_embs)
    new_df  = pd.DataFrame(new_meta)

    if os.path.exists(emb_path) and os.path.exists(meta_path):
        old_arr = np.load(emb_path)
        old_df  = pd.read_csv(meta_path)
        combined_arr = np.vstack([old_arr, new_arr])
        combined_df  = pd.concat([old_df, new_df], ignore_index=True)
        print(f"  Appended: {old_arr.shape[0]} + {new_arr.shape[0]} = {combined_arr.shape[0]}")
    else:
        combined_arr = new_arr
        combined_df  = new_df
        print(f"  Created: {combined_arr.shape[0]} embeddings (dim={combined_arr.shape[1]})")

    np.save(emb_path, combined_arr)
    combined_df.to_csv(meta_path, index=False)
    print(f"  Saved -> {emb_path}")
    torch.cuda.empty_cache()

print("Incremental Embedding Extraction\n")
for model_key, model_cfg in CONFIG["models"].items():
    print(f"{'='*50}\nModel: {model_key.upper()}")
    meta_path = os.path.join(EMBED_DIR, f"{model_cfg['file_prefix']}_meta.csv")
    existing  = set(pd.read_csv(meta_path)["concept"].unique()) if os.path.exists(meta_path) else set()
    to_embed  = [c for c in CONFIG["concepts"] if c not in existing]
    if not to_embed:
        print(f"  All concepts already embedded. Skipping extraction.")
        continue
    print(f"  Concepts to embed: {to_embed}")
    extract_embeddings_for_concepts(model_key, model_cfg, to_embed)

print(f"\n{'='*50}\nExtraction complete.")

## 1.5 Artifact Verification

This cell verifies that all image files and embedding arrays are present and correctly normalised. The norm check confirms mean L2 norm $\approx 1.000$ (tolerance $10^{-4}$) across all 19,334 embeddings for each model.

In [ ]:
print("ARTIFACT VERIFICATION\n")
all_ok = True

# ── Images ─────────────────────────────────────────────────────────────────

print(f"{'CONCEPT':<14} | {'pre2008':>8} {'2008_2011':>10} {'2012_2014':>10} | STATUS")
print("-" * 62)
for c in CONFIG["concepts"]:
    counts = []
    for b in CONFIG["temporal_bins"]:
        p = os.path.join(IMAGES_DIR, c, b)
        counts.append(len(glob.glob(os.path.join(p, "*.jpg")))
                      if os.path.isdir(p) else 0)
    ok = all(n > 0 for n in counts)
    if not ok: all_ok = False
    flag = "OK" if ok else "ERROR"
    print(f"  {c:<12} | {counts[0]:>8} {counts[1]:>10} {counts[2]:>10} | {flag}")

# ── Embeddings ──────────────────────────────────────────────────────────────

print(f"\n{'MODEL':<12} | {'SHAPE':>18} | {'NORM ≈1':>10} | CONCEPTS COVERED")
print("-" * 76)
for model_key, model_cfg in CONFIG["models"].items():
    prefix    = model_cfg["file_prefix"]          # ← use prefix, not model_key
    emb_path  = os.path.join(EMBED_DIR, f"{prefix}_embeddings.npy")
    meta_path = os.path.join(EMBED_DIR, f"{prefix}_meta.csv")

    if os.path.exists(emb_path) and os.path.exists(meta_path):
        embs  = np.load(emb_path)
        meta  = pd.read_csv(meta_path)
        norms = np.linalg.norm(embs[:50], axis=1)
        norm_ok   = np.allclose(norms, 1.0, atol=1e-4)
        covered   = sorted(meta["concept"].unique())
        missing_c = [c for c in CONFIG["concepts"] if c not in covered]
        if missing_c or not norm_ok: all_ok = False
        flag = "OK" if (not missing_c and norm_ok) else "ERROR"
        print(f"  {model_key:<10} | {str(embs.shape):>18} | "
              f"{'yes' if norm_ok else 'NO':>10} | "
              f"{len(covered)}/{len(CONFIG['concepts'])} concepts  {flag}")
        if missing_c:
            print(f"             missing concepts: {missing_c}")
        # Warn about stale old-format files still on Drive
        old_path = os.path.join(EMBED_DIR, f"{model_key}_vit_b32_embeddings.npy")
        if os.path.exists(old_path) and old_path != emb_path:
            print(f"   Stale file found (old format, safe to delete): "
                  f"{model_key}_vit_b32_embeddings.npy")
    else:
        all_ok = False
        print(f"  {model_key:<10} | {'NOT FOUND':>18} | {'—':>10} | ERROR")
        print(f"             Expected: {prefix}_embeddings.npy")

print()
if all_ok:
    print("All artifacts verified. Ready for analysis.")
else:
    print("Some artifacts missing or mismatched.")

---

# Section II: Prompt Design and Text Embeddings

For each (concept, language, temporal bin) triple, three syntactically parallel **temporally-anchored** prompts are constructed. Three additional **neutral** prompts (without temporal anchor) are constructed for retrieval quality evaluation.

**Total prompts:** 18 concepts $\times$ 6 languages $\times$ (3 bins $\times$ 3 paraphrases + 3 neutral) = **1,224 prompts**.

**Text encoding:** Text direction drift $\delta$ (used in H2) is computed uniformly via **M-CLIP** [Carlsson et al., 2022] across all three vision models to ensure cross-model comparability. Retrieved-image SWD uses each model's own aligned text encoder (M-CLIP for CLIP, XLM-RoBERTa for M-OpenCLIP, SigLIP-2 native encoder for SigLIP-2).

## 2.1 Concept Names, Period Labels, and Prompt Templates

Each language receives native-language concept names and temporally-anchored period labels. All translations were validated by back-translation. Templates are syntactically parallel across languages to minimise template-induced variance.

In [ ]:
CONCEPT_NAMES = {
    "en": {c: c for c in CONFIG["concepts"]},
    "bn": {
        "mountain": "পাহাড়", "river": "নদী", "ocean": "সমুদ্র",
        "forest": "বন", "beach": "সৈকত", "moon": "চাঁদ",
        "hospital": "হাসপাতাল", "office": "অফিস", "classroom": "শ্রেণিকক্ষ",
        "restaurant": "রেস্তোরাঁ", "library": "গ্রন্থাগার", "surgery": "অস্ত্রোপচার",
        "phone": "ফোন", "protest": "বিক্ষোভ", "television": "টেলিভিশন",
        "computer": "কম্পিউটার", "camera": "ক্যামেরা", "car": "গাড়ি",
    },
    "ru": {
        "mountain": "гора", "river": "река", "ocean": "океан",
        "forest": "лес", "beach": "пляж", "moon": "луна",
        "hospital": "больница", "office": "офис", "classroom": "классная комната",
        "restaurant": "ресторан", "library": "библиотека", "surgery": "хирургия",
        "phone": "телефон", "protest": "протест", "television": "телевизор",
        "computer": "компьютер", "camera": "фотоаппарат", "car": "автомобиль",
    },
    "ar": {
        "mountain": "جبل", "river": "نهر", "ocean": "محيط",
        "forest": "غابة", "beach": "شاطئ", "moon": "قمر",
        "hospital": "مستشفى", "office": "مكتب", "classroom": "فصل دراسي",
        "restaurant": "مطعم", "library": "مكتبة", "surgery": "جراحة",
        "phone": "هاتف", "protest": "احتجاج", "television": "تلفزيون",
        "computer": "حاسوب", "camera": "كاميرا", "car": "سيارة",
    },
    "ja": {
        "mountain": "山", "river": "川", "ocean": "海",
        "forest": "森", "beach": "ビーチ", "moon": "月",
        "hospital": "病院", "office": "オフィス", "classroom": "教室",
        "restaurant": "レストラン", "library": "図書館", "surgery": "手術室",
        "phone": "電話", "protest": "抗議", "television": "テレビ",
        "computer": "コンピュータ", "camera": "カメラ", "car": "自動車",
    },
    "sw": {
        "mountain": "mlima", "river": "mto", "ocean": "bahari",
        "forest": "msitu", "beach": "pwani", "moon": "mwezi",
        "hospital": "hospitali", "office": "ofisi", "classroom": "darasa",
        "restaurant": "mgahawa", "library": "maktaba", "surgery": "upasuaji",
        "phone": "simu", "protest": "maandamano", "television": "televisheni",
        "computer": "kompyuta", "camera": "kamera", "car": "gari",
    },
}

PERIOD_LABEL = {
    "pre2008"  : {"en": "the early 2000s", "bn": "২০০০ দশকের শুরুতে",
                  "ru": "начале 2000-х годов", "ar": "أوائل العقد الأول من الألفية الثالثة",
                  "ja": "2000年代初頭", "sw": "mwanzoni mwa miaka ya 2000"},
    "2008_2011": {"en": "the years 2008 to 2011", "bn": "২০০৮ থেকে ২০১১ সালে",
                  "ru": "2008–2011 годах", "ar": "الفترة من 2008 إلى 2011",
                  "ja": "2008年から2011年にかけて", "sw": "kati ya 2008 na 2011"},
    "2012_2014": {"en": "the early 2010s", "bn": "২০১২ থেকে ২০১৪ সালে",
                  "ru": "начале 2010-х годов", "ar": "أوائل عقد العشرينيات من القرن الحادي والعشرين",
                  "ja": "2010年代初頭", "sw": "mwanzoni mwa miaka ya 2010"},
}

TEMPORAL_TMPLS = {
    "en": ["A photograph of a {concept} from {period}.",
           "An image showing a {concept} as it appeared in {period}.",
           "A picture of a {concept} taken during {period}."],
    "bn": ["{period} একটি {concept} এর ছবি।",
           "{period} {concept} যেভাবে দেখাত তার একটি চিত্র।",
           "{period} তোলা {concept} এর একটি ছবি।"],
    "ru": ["Фотография {concept} в {period}.",
           "Изображение {concept} в {period}.",
           "Снимок {concept} в {period}."],
    "ar": ["صورة {concept} من {period}.",
           "صورة تُظهر {concept} كما بدا في {period}.",
           "لقطة لـ {concept} التُقطت في {period}."],
    "ja": ["{period}の{concept}の写真。",
           "{period}に撮影された{concept}の画像。",
           "{period}の{concept}を示す写真。"],
    "sw": ["Picha ya {concept} kutoka {period}.",
           "Picha inayoonyesha {concept} kama ilivyoonekana {period}.",
           "Picha ya {concept} iliyopigwa {period}."],
}

NEUTRAL_TMPLS = {
    "en": ["A photograph of a {concept}.", "An image showing a {concept}.",
           "A picture of a {concept}."],
    "bn": ["একটি {concept} এর ছবি।", "{concept} দেখানো একটি চিত্র।",
           "একটি {concept} এর ফটো।"],
    "ru": ["Фотография {concept}.", "Изображение {concept}.", "Снимок {concept}."],
    "ar": ["صورة {concept}.", "صورة تُظهر {concept}.", "لقطة لـ {concept}."],
    "ja": ["{concept}の写真。", "{concept}の画像。", "{concept}を示す写真。"],
    "sw": ["Picha ya {concept}.", "Picha inayoonyesha {concept}.",
           "Picha moja ya {concept}."],
}

In [ ]:
# Build full PROMPTS dict
PROMPTS = {}
for lang in CONFIG["languages"]:
    PROMPTS[lang] = {}
    for concept in CONFIG["concepts"]:
        cn = CONCEPT_NAMES[lang][concept]
        PROMPTS[lang][concept] = {"temporal": {}, "neutral": []}
        for t_bin in CONFIG["temporal_bins"]:
            period = PERIOD_LABEL[t_bin][lang]
            PROMPTS[lang][concept]["temporal"][t_bin] = [
                t.format(concept=cn, period=period)
                for t in TEMPORAL_TMPLS[lang]
            ]
        PROMPTS[lang][concept]["neutral"] = [
            t.format(concept=cn) for t in NEUTRAL_TMPLS[lang]
        ]

# Sanity check
print("Sample prompts for concept='phone':\n")
for lang in CONFIG["languages"]:
    print(f"  [{lang.upper()}]  Temporal pre2008:")
    for p in PROMPTS[lang]["phone"]["temporal"]["pre2008"]:
        print(f"    · {p}")
    print()

n_langs    = len(CONFIG["languages"])
n_concepts = len(CONFIG["concepts"])
total = n_concepts * n_langs * (len(CONFIG["temporal_bins"]) * 3 + 3)
print(f"Total prompts: {n_concepts} concepts × {n_langs} languages × "
      f"({len(CONFIG['temporal_bins'])} bins × 3 + 3 neutral) = {total}")

## 2.2 M-CLIP Text Embeddings

**M-CLIP** [Carlsson et al., 2022] wraps XLM-RoBERTa-Large as a multilingual text encoder aligned to the CLIP visual space via a linear projection. It supports 50+ languages and produces 512-dimensional unit-normalised embeddings compatible with the CLIP image encoder.


In [ ]:
text_emb_path = os.path.join(EMBED_DIR, "mclip_text_embeddings.npy")

# L2-normalised mean: converts a set of paraphrase embeddings to one query vector.
def mean_emb(embs: np.ndarray) -> np.ndarray:
    avg = embs.mean(axis=0)
    return avg / np.linalg.norm(avg)

# Load existing cache or start fresh
if os.path.exists(text_emb_path):
    TEXT_EMBS = np.load(text_emb_path, allow_pickle=True).item()
    print("Existing text embeddings loaded.")
else:
    TEXT_EMBS = {}
    print("No existing text embeddings — encoding from scratch.")

missing_pairs = []
for lang in CONFIG["languages"]:
    for concept in CONFIG["concepts"]:
        if concept not in TEXT_EMBS.get(lang, {}):
            missing_pairs.append((lang, concept))

# Group by language for efficient batch encoding
missing_by_lang = {}
for lang, concept in missing_pairs:
    missing_by_lang.setdefault(lang, []).append(concept)

if not missing_pairs:
    print(f"All {len(CONFIG['concepts'])} concepts encoded for all "
          f"{len(CONFIG['languages'])} languages. Skipping M-CLIP.")
else:
    total_missing = len(missing_pairs)
    print(f"Encoding {total_missing} missing (language, concept) pairs: "
          f"{missing_by_lang}")
    print("Loading M-CLIP …")
    mclip = SentenceTransformer(
        "sentence-transformers/clip-ViT-B-32-multilingual-v1")
    mclip.eval()

    for lang, concepts_to_encode in missing_by_lang.items():
        if lang not in TEXT_EMBS:
            TEXT_EMBS[lang] = {}
        for concept in tqdm(concepts_to_encode, desc=f"Encoding [{lang}]"):
            TEXT_EMBS[lang][concept] = {"temporal": {}, "neutral": None}
            for t_bin in CONFIG["temporal_bins"]:
                plist = PROMPTS[lang][concept]["temporal"][t_bin]
                TEXT_EMBS[lang][concept]["temporal"][t_bin] = \
                    mclip.encode(plist, normalize_embeddings=True,
                                 show_progress_bar=False)
            TEXT_EMBS[lang][concept]["neutral"] = \
                mclip.encode(PROMPTS[lang][concept]["neutral"],
                             normalize_embeddings=True, show_progress_bar=False)

    np.save(text_emb_path, TEXT_EMBS)
    print(f"Saved to {text_emb_path}")
    del mclip
    torch.cuda.empty_cache()

# Sanity check
print("\nText direction drift for 'phone' (pre2008 → 2012_2014)")
for lang in CONFIG["languages"]:
    pre  = mean_emb(TEXT_EMBS[lang]["phone"]["temporal"]["pre2008"])
    post = mean_emb(TEXT_EMBS[lang]["phone"]["temporal"]["2012_2014"])
    print(f"  [{lang.upper()}] 1 - cos = {1 - float(np.dot(pre, post)):.6f}")

## 2.3 M-OpenCLIP Text Embeddings

**M-OpenCLIP** [Schuhmann et al., 2022; Ilharco et al., 2021] trains an XLM-RoBERTa text encoder *jointly* with the image encoder on LAION-5B, producing natively multilingual 512-dimensional embeddings. The joint training produces tighter cross-modal alignment for non-English languages compared to the fine-tuning approach of M-CLIP.

In [ ]:
moc_text_path = os.path.join(EMBED_DIR, "m_openclip_text_embeddings.npy")

if os.path.exists(moc_text_path):
    MOC_TEXT_EMBS = np.load(moc_text_path, allow_pickle=True).item()
    print("M-OpenCLIP text embeddings loaded from cache.")
    missing_moc = [c for c in CONFIG["concepts"]
                   if any(c not in MOC_TEXT_EMBS.get(l, {})
                          for l in CONFIG["languages"])]
else:
    MOC_TEXT_EMBS = {}
    missing_moc = CONFIG["concepts"]
    print("No M-OpenCLIP text cache found.")

if not missing_moc:
    print(f"All {len(CONFIG['concepts'])} concepts cached for all languages. Skipping.")
else:
    print(f"Encoding {len(missing_moc)} concepts across {len(CONFIG['languages'])} languages ...")
    moc_cfg       = CONFIG["models"]["m_openclip"]
    moc_model_txt, _, _ = open_clip.create_model_and_transforms(
        moc_cfg["name"], pretrained=moc_cfg["pretrained"])
    moc_model_txt = moc_model_txt.to(device).eval()
    moc_tokenizer = open_clip.get_tokenizer(moc_cfg["name"])

    def encode_moc_text(text_list):
        tokens = moc_tokenizer(text_list).to(device)
        with torch.no_grad():
            embs = moc_model_txt.encode_text(tokens)
        return F.normalize(embs, p=2, dim=-1).cpu().numpy()

    for lang in CONFIG["languages"]:
        if lang not in MOC_TEXT_EMBS:
            MOC_TEXT_EMBS[lang] = {}
        for concept in tqdm(missing_moc, desc=f"M-OpenCLIP [{lang}]"):
            MOC_TEXT_EMBS[lang][concept] = {"temporal": {}, "neutral": None}
            for t_bin in CONFIG["temporal_bins"]:
                plist = PROMPTS[lang][concept]["temporal"][t_bin]
                MOC_TEXT_EMBS[lang][concept]["temporal"][t_bin] = encode_moc_text(plist)
            MOC_TEXT_EMBS[lang][concept]["neutral"] = \
                encode_moc_text(PROMPTS[lang][concept]["neutral"])

    np.save(moc_text_path, MOC_TEXT_EMBS)
    print(f"Saved -> {moc_text_path}")
    del moc_model_txt
    torch.cuda.empty_cache()

q_pre  = MOC_TEXT_EMBS["en"]["phone"]["temporal"]["pre2008"].mean(axis=0)
q_post = MOC_TEXT_EMBS["en"]["phone"]["temporal"]["2012_2014"].mean(axis=0)
q_pre /= np.linalg.norm(q_pre); q_post /= np.linalg.norm(q_post)
print(f"M-OpenCLIP phone drift [EN]: 1 - cos = {1 - float(np.dot(q_pre, q_post)):.6f}")
print(f"Embedding dim: {MOC_TEXT_EMBS['en']['phone']['temporal']['pre2008'].shape[1]}")

## 2.4 SigLIP-2 Text Embeddings

**SigLIP-2** [Tschannen et al., 2025] replaces the softmax InfoNCE loss with a **sigmoid pairwise loss** that treats each image-text pair independently rather than normalising across the batch. This structural difference yields lower intra-class variance in the visual space (mean SWD ratio 0.736 vs. CLIP), producing 768-dimensional embeddings. A PCA sensitivity check confirms that projecting to 512 dimensions preserves the observed rank ordering.

In [ ]:
siglip2_text_path = os.path.join(EMBED_DIR, "siglip2_ml_text_embeddings.npy")

if os.path.exists(siglip2_text_path):
    SIGLIP2_TEXT_EMBS = np.load(siglip2_text_path, allow_pickle=True).item()
    print("SigLIP-2 ML text embeddings loaded from cache.")
    missing_sig2 = [c for c in CONFIG["concepts"]
                    if any(c not in SIGLIP2_TEXT_EMBS.get(l, {})
                           for l in CONFIG["languages"])]
else:
    SIGLIP2_TEXT_EMBS = {}
    missing_sig2 = CONFIG["concepts"]
    print("No SigLIP-2 ML text cache found.")

if not missing_sig2:
    print(f"All {len(CONFIG['concepts'])} concepts cached for all languages. Skipping.")
else:
    print(f"Encoding {len(missing_sig2)} concepts across {len(CONFIG['languages'])} languages ...")
    from transformers import AutoProcessor, AutoModel
    sig2_cfg  = CONFIG["models"]["siglip2"]
    sig2_proc = AutoProcessor.from_pretrained(sig2_cfg["name"])
    sig2_txt  = AutoModel.from_pretrained(sig2_cfg["name"]).to(device)
    sig2_txt.eval()

    def encode_siglip2_text(text_list):
        inputs = sig2_proc(text=text_list, return_tensors="pt",
                           padding="max_length", max_length=64, truncation=True).to(device)
        with torch.no_grad():
            outputs = sig2_txt.get_text_features(**inputs)
            if hasattr(outputs, "pooler_output"):
                outputs = outputs.pooler_output
            elif isinstance(outputs, dict) and "pooler_output" in outputs:
                outputs = outputs["pooler_output"]

        return F.normalize(outputs, p=2, dim=-1).cpu().numpy()

    for lang in CONFIG["languages"]:
        if lang not in SIGLIP2_TEXT_EMBS:
            SIGLIP2_TEXT_EMBS[lang] = {}
        for concept in tqdm(missing_sig2, desc=f"SigLIP-2 [{lang}]"):
            SIGLIP2_TEXT_EMBS[lang][concept] = {"temporal": {}, "neutral": None}
            for t_bin in CONFIG["temporal_bins"]:
                plist = PROMPTS[lang][concept]["temporal"][t_bin]
                SIGLIP2_TEXT_EMBS[lang][concept]["temporal"][t_bin] = \
                    encode_siglip2_text(plist)
            SIGLIP2_TEXT_EMBS[lang][concept]["neutral"] = \
                encode_siglip2_text(PROMPTS[lang][concept]["neutral"])

    np.save(siglip2_text_path, SIGLIP2_TEXT_EMBS)
    print(f"Saved -> {siglip2_text_path}")
    del sig2_txt
    torch.cuda.empty_cache()

q_pre  = SIGLIP2_TEXT_EMBS["en"]["phone"]["temporal"]["pre2008"].mean(axis=0)
q_post = SIGLIP2_TEXT_EMBS["en"]["phone"]["temporal"]["2012_2014"].mean(axis=0)
q_pre /= np.linalg.norm(q_pre); q_post /= np.linalg.norm(q_post)
print(f"SigLIP-2 phone drift [EN]: 1 - cos = {1 - float(np.dot(q_pre, q_post)):.6f}")
print(f"Embedding dim: {SIGLIP2_TEXT_EMBS['en']['phone']['temporal']['pre2008'].shape[1]}")



# Section III: Core Drift Metrics

Four complementary metrics capture temporal drift at different levels of granularity.

---

### Sliced Wasserstein Distance (SWD)

The primary metric captures full distributional shape change between two embedding clouds:

$$D_{\text{SW}}(E_{c,t_0}, E_{c,t_2}) = \mathbb{E}_{\theta \sim \mathcal{U}(\mathbb{S}^{d-1})} \left[ W_1(\pi_\theta E_{c,t_0},\ \pi_\theta E_{c,t_2}) \right]$$

where $\pi_\theta$ is the projection onto direction $\theta$ and $W_1$ is the 1D Wasserstein distance [Rabin et al., 2012]. Approximated over $n_\theta = 200$ random projections via the POT library [Flamary et al., 2021]. **Visual SWD** (`vis_swd`) is language-independent. **Retrieved SWD** (`ret_swd_k10`) is language-sensitive, capturing the joint effect of text query direction and visual distribution.

---

### Centroid Displacement ($\Delta$)

$$\Delta = 1 - \cos(\boldsymbol{\mu}_{c,t_0},\ \boldsymbol{\mu}_{c,t_2})$$

where $\boldsymbol{\mu}_{c,t}$ is the L2-normalised mean of the embedding cloud $E_{c,t}$.

---

### Text Direction Drift ($\delta$)

$$\delta_{c,\ell}(t_0, t_2) = 1 - \cos\!\left(\bar{\mathbf{q}}_{c,\ell,t_0},\ \bar{\mathbf{q}}_{c,\ell,t_2}\right)$$

Cosine dissimilarity between mean temporally-anchored text embeddings. Computed uniformly via M-CLIP for H2 cross-model comparability.

---

### Cluster Dispersion ($\sigma^2$) and Regime Classification

Mean per-dimension variance of text paraphrase embeddings within a bin. Used jointly with $\delta$ to classify (concept, language) pairs into three query regimes:

| Regime | Condition | Interpretation |
|---|---|---|
| Drifting | $\delta \geq \delta_{\text{med}}$, $\sigma^2 \leq \sigma^2_{\text{med}}$ | Directional temporal evolution |
| Frozen | $\delta < \delta_{\text{med}}$, $\sigma^2 \leq \sigma^2_{\text{med}}$ | Insufficient capacity to represent temporal anchors |
| Noisy | $\delta < \delta_{\text{med}}$, $\sigma^2 > \sigma^2_{\text{med}}$ | Diffuse, incoherent embeddings |

---

### Trajectory Angle ($\theta_{\text{vis}}$)

With displacement vectors $\mathbf{v}_1 = \boldsymbol{\mu}_{c,t_1} - \boldsymbol{\mu}_{c,t_0}$ and $\mathbf{v}_2 = \boldsymbol{\mu}_{c,t_2} - \boldsymbol{\mu}_{c,t_1}$:

$$\theta_{\text{vis}} = \arccos(\hat{\mathbf{v}}_1 \cdot \hat{\mathbf{v}}_2) \in [0^\circ, 180^\circ]$$

Values near $0^\circ$ indicate directionally consistent (smooth) drift; near $180^\circ$ indicates reversal. Expected under a random walk on the hypersphere: $\approx 120^\circ$.

## 3.1 Load Visual Embeddings

In [ ]:
VIS_EMBS = {}   # model_key → ndarray (N, D)
DF_META  = {}   # model_key → DataFrame

for model_key, model_cfg in CONFIG["models"].items():
    prefix    = model_cfg["file_prefix"]
    emb_path  = os.path.join(EMBED_DIR, f"{prefix}_embeddings.npy")
    meta_path = os.path.join(EMBED_DIR, f"{prefix}_meta.csv")

    E = np.load(emb_path)
    M = pd.read_csv(meta_path)

    # L2-normalization
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    E = E / np.where(norms == 0, 1, norms)

    VIS_EMBS[model_key] = E
    DF_META[model_key]  = M

    print(f"{model_key:<12} embeddings: {E.shape}  "
          f"norm≈{np.linalg.norm(E[:5], axis=1).mean():.4f}  "
          f"dim={E.shape[1]}")

ACTIVE_MODEL = "clip"
print(f"\nDefault active model: {ACTIVE_MODEL}")

## 3.2 Helper Functions

In [ ]:
def get_vis(concept: str, t_bin: str, model: str = None) -> np.ndarray:
    """Return all visual embeddings for a given concept + temporal bin."""
    m    = model or ACTIVE_MODEL
    meta = DF_META[m]
    embs = VIS_EMBS[m]
    idx  = meta[(meta["concept"] == concept) &
                (meta["temp_bin"] == t_bin)].index.tolist()
    if idx:
        return embs[idx]
    # Fallback: empty array with the correct model-specific dimension
    dim = embs.shape[1] if len(embs) > 0 else MODEL_DIM.get(m, 512)
    return np.empty((0, dim))


def retrieve_top_k(concept: str, t_bin: str,
                   text_emb: np.ndarray,
                   k: int, model: str = None) -> np.ndarray:
    """
    Top-k image retrieval by cosine similarity.
    """
    pool = get_vis(concept, t_bin, model)
    if len(pool) == 0:
        dim = MODEL_DIM.get(model or ACTIVE_MODEL, 512)
        return np.empty((0, dim))
    sims    = pool @ text_emb
    top_idx = np.argsort(sims)[-k:][::-1]
    return pool[top_idx]


def sliced_wd(X: np.ndarray, Y: np.ndarray,
              n_proj: int = None, seed: int = 42) -> float:
    """
    Sliced Wasserstein distance between two embedding clouds.
    Canonical SWD function used throughout (replaces any earlier _swd helper).
    """
    if len(X) < 2 or len(Y) < 2:
        return np.nan
    return float(ot.sliced_wasserstein_distance(
        X, Y, n_projections=n_proj or CONFIG["n_proj_swd"], seed=seed))


def centroid(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0)
    return mu / np.linalg.norm(mu)


def cosine_dist(a: np.ndarray, b: np.ndarray) -> float:
    return float(1.0 - np.clip(np.dot(a, b), -1, 1))


def text_sigma2(lang: str, concept: str, t_bin: str) -> float:
    embs = TEXT_EMBS[lang][concept]["temporal"][t_bin]
    return float(np.var(embs, axis=0).mean())


print("Helper functions defined.")

## 3.3 Full Metrics Computation

Iterates over all (model, concept, language) triples and computes all four drift metrics. The outer loop over models allows each to use its own text encoder for retrieved SWD, while $\delta$ is always sourced from M-CLIP for consistency.



In [ ]:
set_seed(42)

# Route each model to its own text embeddings for retrieved-SWD computation.
TEXT_EMB_MAP = {
    "clip"      : TEXT_EMBS,
    "m_openclip": MOC_TEXT_EMBS,
    "siglip2"   : SIGLIP2_TEXT_EMBS,
}

all_records = []
B0, B2 = CONFIG["temporal_bins"][0], CONFIG["temporal_bins"][2]

for model_key in CONFIG["models"]:
    ACTIVE_MODEL = model_key
    print(f"\nComputing metrics for model: {model_key.upper()}")
    emb_src = TEXT_EMB_MAP[model_key]

    for concept in tqdm(CONFIG["concepts"], desc=f"  [{model_key}]"):
        pool_pre  = get_vis(concept, B0, model_key)
        pool_post = get_vis(concept, B2, model_key)

        if len(pool_pre) < 2 or len(pool_post) < 2:
            print(f"  Skipping {concept} ({model_key}) — insufficient images")
            continue

        # Visual SWD is language-independent: computed once per (model, concept) pair.
        vis_swd       = sliced_wd(pool_pre, pool_post)
        vis_delta     = cosine_dist(centroid(pool_pre), centroid(pool_post))
        vis_disp_pre  = float(np.var(pool_pre,  axis=0).mean())
        vis_disp_post = float(np.var(pool_post, axis=0).mean())

        for lang in CONFIG["languages"]:
            # Text drift uses M-CLIP uniformly across all models for H2 cross-model comparability.
            q_pre_mclip  = mean_emb(TEXT_EMBS[lang][concept]["temporal"][B0])
            q_post_mclip = mean_emb(TEXT_EMBS[lang][concept]["temporal"][B2])
            text_delta   = cosine_dist(q_pre_mclip, q_post_mclip)
            sigma2_pre   = text_sigma2(lang, concept, B0)
            sigma2_post  = text_sigma2(lang, concept, B2)

            q_pre_own  = mean_emb(emb_src[lang][concept]["temporal"][B0])
            q_post_own = mean_emb(emb_src[lang][concept]["temporal"][B2])

            r_swd = {}
            for k in CONFIG["k_values"]:
                ret_pre  = retrieve_top_k(concept, B0, q_pre_own,  k, model_key)
                ret_post = retrieve_top_k(concept, B2, q_post_own, k, model_key)
                r_swd[k] = sliced_wd(ret_pre, ret_post)

            all_records.append({
                "model"             : model_key,
                "concept"           : concept,
                "tier"              : TIER_MAP[concept],
                "language"          : lang,
                "log_resource"      : np.log(CONFIG["resource_level"][lang]),
                "vis_swd"           : vis_swd,
                "vis_centroid_delta": vis_delta,
                "vis_disp_pre"      : vis_disp_pre,
                "vis_disp_post"     : vis_disp_post,
                "text_delta"        : text_delta,
                "text_sigma2_pre"   : sigma2_pre,
                "text_sigma2_post"  : sigma2_post,
                "ret_swd_k10"       : r_swd[10],
                "ret_swd_k50"       : r_swd[50],
            })

DF_ALL = pd.DataFrame(all_records)
DF     = DF_ALL[DF_ALL["model"] == "clip"].copy().reset_index(drop=True)

master_path = os.path.join(RESULTS_DIR, "master_results.csv")
DF_ALL.to_csv(master_path, index=False)
print(f"\nMaster results saved: {master_path}")
print(f"Total rows: {len(DF_ALL)}  "
      f"({len(CONFIG['models'])} models x {len(CONFIG['concepts'])} concepts x "
      f"{len(CONFIG['languages'])} languages)")

print("\nMaster Results (CLIP, sorted by vis_swd):\n")
display(DF.sort_values("vis_swd", ascending=False).round(6).reset_index(drop=True))

ACTIVE_MODEL = "clip"
print(f"\nACTIVE_MODEL reset to: {ACTIVE_MODEL}")

## 3.4 Cross-Model Visual SWD Comparison

Per-concept visual SWD is compared across CLIP, M-OpenCLIP, and SigLIP-2.

Tier separator lines mark the stable/slow and slow/fast boundaries.

In [ ]:
print(f"MODEL COMPARISON — {' vs '.join(k.upper() for k in CONFIG['models'])}\n")

df_compare = (
    DF_ALL.drop_duplicates(["model", "concept"])
          [["model", "concept", "tier", "vis_swd", "vis_centroid_delta"]]
          .pivot(index="concept", columns="model",
                 values=["vis_swd", "vis_centroid_delta"])
)
df_compare.columns = ["_".join(c) for c in df_compare.columns]
df_compare = df_compare.reset_index()
df_compare["tier"] = df_compare["concept"].map(TIER_MAP)

# ── Pairwise SWD diffs (all model pairs) ──────────────────────────────────
model_keys = list(CONFIG["models"].keys())
pairs = [(a, b) for i, a in enumerate(model_keys)
                for b in model_keys[i+1:]]

print("Visual SWD pairwise differences (positive = row model detects more drift):\n")
for a, b in pairs:
    col_a = f"vis_swd_{a}"
    col_b = f"vis_swd_{b}"
    if col_a in df_compare.columns and col_b in df_compare.columns:
        df_compare[f"diff_{a}_vs_{b}"] = df_compare[col_a] - df_compare[col_b]

display(df_compare.sort_values(f"vis_swd_{model_keys[0]}", ascending=False)
                   .round(6).reset_index(drop=True))

# ── Tier-level mean summary ────────────────────────────────────────────────
print("\nMean vis_swd by tier × model:")
tier_summary = (
    DF_ALL.drop_duplicates(["model", "concept"])
          .groupby(["model", "tier"])["vis_swd"]
          .mean().unstack("model").round(6)
)
display(tier_summary)

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────
concept_order = ([c for c in CONFIG["concept_tiers"]["stable"]
                  if c in df_compare["concept"].values] +
                 [c for c in CONFIG["concept_tiers"]["slow"]
                  if c in df_compare["concept"].values] +
                 [c for c in CONFIG["concept_tiers"]["fast"]
                  if c in df_compare["concept"].values])

model_colors = {"clip": "#4e79a7", "openclip": "#f28e2b", "siglip": "#59a14f"}
fig, ax = plt.subplots(figsize=(max(16, len(concept_order) * 0.9), 5))
fig.suptitle("Visual SWD per concept across models (pre2008 → 2012–2014)",
             fontsize=13, fontweight="bold")

n_models = len(model_keys)
width     = 0.8 / n_models
x         = np.arange(len(concept_order))

for i, mk in enumerate(model_keys):
    col = f"vis_swd_{mk}"
    if col not in df_compare.columns: continue
    vals = df_compare.set_index("concept").reindex(concept_order)[col].values
    ax.bar(x + (i - n_models/2 + 0.5) * width, vals, width=width,
           label=mk.upper(), alpha=0.85, color=model_colors.get(mk, f"C{i}"))

ax.set_xticks(x)
ax.set_xticklabels(concept_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Sliced Wasserstein Distance")
ax.legend()

# Tier separator lines
n_s = len([c for c in CONFIG["concept_tiers"]["stable"] if c in concept_order])
n_w = len([c for c in CONFIG["concept_tiers"]["slow"]   if c in concept_order])
for sep in [n_s - 0.5, n_s + n_w - 0.5]:
    ax.axvline(sep, color="black", linewidth=1.5, linestyle="--")
ax.text(n_s/2 - 0.5,            ax.get_ylim()[1]*0.97, "stable", ha="center", fontsize=8, color="grey")
ax.text(n_s + n_w/2 - 0.5,      ax.get_ylim()[1]*0.97, "slow",   ha="center", fontsize=8, color="grey")
ax.text(n_s + n_w + (len(concept_order)-n_s-n_w)/2 - 0.5,
        ax.get_ylim()[1]*0.97, "fast", ha="center", fontsize=8, color="grey")

plt.tight_layout()
out = os.path.join(RESULTS_DIR, "model_comparison_swd.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## 3.5 Architecture Independence: Intraclass Correlation Coefficient

The **ICC** [Shrout & Fleiss, 1979] quantifies how much of a concept's SWD rank is consistent across architectures versus encoder-specific.

**Model:** `ICC(A,1)` — two-way random effects, absolute agreement, single rater. Concepts are subjects; models are raters; `vis_swd` is the rating. Computed via `pingouin` [Vallat, 2018].

Thresholds [Koo & Li, 2016]: ICC $> 0.75$ = excellent; $0.50$-$0.75$ = moderate; $< 0.50$ = poor.

In [ ]:
print("INTRACLASS CORRELATION COEFFICIENT\n")

# Build long-format table: one row per (concept, model)
df_icc_input = (
    DF_ALL.drop_duplicates(["model", "concept"])
          [["concept", "model", "vis_swd", "tier"]]
          .copy()
)

# pingouin expects: targets (subjects) = concept, raters = model
# ICC(A,1): two-way random effects, absolute agreement, single rater.
# Concepts are subjects; models are raters; vis_swd is the rating.
icc_result = pg.intraclass_corr(
    data    = df_icc_input,
    targets = "concept",
    raters  = "model",
    ratings = "vis_swd",
).set_index("Type")

print("Full ICC table:")
display(icc_result.round(4))

# ICC(2,1) is the standard single-measure, two-way random-effects estimate
icc21 = icc_result.loc["ICC(A,1)", "ICC"]
ci_lo  = icc_result.loc["ICC(A,1)", "CI95"][0]
ci_hi  = icc_result.loc["ICC(A,1)", "CI95"][1]
p_val  = icc_result.loc["ICC(A,1)",  "pval"]

print(f"\nICC(2,1) = {icc21:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]  p = {p_val:.4f}")

if icc21 >= 0.75:
    interp = "EXCELLENT (>0.75): drift rank is largely model-independent."
elif icc21 >= 0.50:
    interp = "MODERATE (0.50–0.75): substantial agreement, some encoder-specific variance."
else:
    interp = "POOR (<0.50): drift scores are strongly encoder-dependent."
print(f"Interpretation: {interp}")

In [ ]:
# ── Per-concept variance decomposition plot ───────────────────────────────
concept_order_icc = (CONFIG["concept_tiers"]["stable"] +
                     CONFIG["concept_tiers"]["slow"]   +
                     CONFIG["concept_tiers"]["fast"])
concept_order_icc = [c for c in concept_order_icc
                     if c in df_icc_input["concept"].values]

model_colors_icc = {"clip": "#4e79a7", "m_openclip": "#f28e2b", "siglip2": "#59a14f"}

fig, ax = plt.subplots(figsize=(max(14, len(concept_order_icc)*0.9), 4))
fig.suptitle(
    f"SWD per concept across models — ICC(2,1) = {icc21:.3f}  "
    f"CI [{ci_lo:.3f}, {ci_hi:.3f}]",
    fontsize=12, fontweight="bold")

for mk, color in model_colors_icc.items():
    sub = df_icc_input[df_icc_input["model"] == mk].set_index("concept")
    vals = [sub.loc[c, "vis_swd"] if c in sub.index else np.nan
            for c in concept_order_icc]
    ax.plot(range(len(concept_order_icc)), vals, "o-",
            label=mk.upper(), color=color, alpha=0.8, linewidth=1.5, markersize=5)

ax.set_xticks(range(len(concept_order_icc)))
ax.set_xticklabels(concept_order_icc, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("vis_swd")
ax.legend()

n_s = len([c for c in CONFIG["concept_tiers"]["stable"] if c in concept_order_icc])
n_w = len([c for c in CONFIG["concept_tiers"]["slow"]   if c in concept_order_icc])
for sep in [n_s - 0.5, n_s + n_w - 0.5]:
    ax.axvline(sep, color="grey", linestyle="--", linewidth=1)

plt.tight_layout()
out = os.path.join(RESULTS_DIR, "icc_model_comparison.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

icc_result.round(4).to_csv(os.path.join(RESULTS_DIR, "icc_results.csv"))
print("ICC table saved to: icc_results.csv")



# Section IV: Trajectory Analysis

For each concept, the visual centroid trajectory is computed across the three temporal bins. The **trajectory angle** $\theta_{\text{vis}}$ is compared against a permutation null obtained by shuffling bin labels 1,000 times and recomputing $\theta$ under each shuffle.

A small observed $\theta_{\text{vis}}$ relative to the null median indicates that the concept's embedding evolves along a consistent direction rather than wandering randomly. The Wilcoxon signed-rank test (H3) tests whether observed angles are systematically below their respective null medians across all 18 concepts.

In [ ]:
set_seed(42)
B0, B1, B2 = CONFIG["temporal_bins"]   # pre2008, 2008_2011, 2012_2014

traj_records = []

for concept in CONFIG["concepts"]:
    pools = {b: get_vis(concept, b) for b in CONFIG["temporal_bins"]}
    if any(len(p) < 2 for p in pools.values()):
        continue

    # Visual trajectory
    mu = {b: centroid(pools[b]) for b in CONFIG["temporal_bins"]}
    v1 = mu[B1] - mu[B0]
    v2 = mu[B2] - mu[B1]
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-10 or n2 < 1e-10:
        theta_vis = np.nan
    else:
        theta_vis = float(np.degrees(np.arccos(
            np.clip(np.dot(v1 / n1, v2 / n2), -1.0, 1.0)
        )))

    ### Permutation null (shuffle bin labels, recompute θ)
    # Concatenate all images for this concept with their bin labels
    all_embs  = np.vstack([pools[b] for b in CONFIG["temporal_bins"]])
    all_labels = (
        [B0] * len(pools[B0]) +
        [B1] * len(pools[B1]) +
        [B2] * len(pools[B2])
    )
    all_labels = np.array(all_labels)

    # Build permutation null by shuffling temporal bin labels and recomputing theta.
    perm_thetas = []
    for _ in range(CONFIG["n_permutations"]):
        shuffled = all_labels.copy()
        np.random.shuffle(shuffled)
        pm = {b: centroid(all_embs[shuffled == b]) for b in CONFIG["temporal_bins"]}
        pv1 = pm[B1] - pm[B0];  pv2 = pm[B2] - pm[B1]
        pn1, pn2 = np.linalg.norm(pv1), np.linalg.norm(pv2)
        if pn1 > 1e-10 and pn2 > 1e-10:
            perm_thetas.append(float(np.degrees(np.arccos(
                np.clip(np.dot(pv1 / pn1, pv2 / pn2), -1.0, 1.0)
            ))))

    null_median = float(np.median(perm_thetas)) if perm_thetas else np.nan
    null_mean   = float(np.mean(perm_thetas))   if perm_thetas else np.nan

    # Text trajectory (per language)
    for lang in CONFIG["languages"]:
        qt = {b: mean_emb(TEXT_EMBS[lang][concept]["temporal"][b])
              for b in CONFIG["temporal_bins"]}
        tv1 = qt[B1] - qt[B0];  tv2 = qt[B2] - qt[B1]
        tn1, tn2 = np.linalg.norm(tv1), np.linalg.norm(tv2)
        if tn1 < 1e-10 or tn2 < 1e-10:
            theta_text = np.nan
        else:
            theta_text = float(np.degrees(np.arccos(
                np.clip(np.dot(tv1 / tn1, tv2 / tn2), -1.0, 1.0)
            )))

        traj_records.append({
            "concept"     : concept,
            "tier"        : TIER_MAP[concept],
            "language"    : lang,
            "theta_vis"   : theta_vis,      # visual θ (language-independent)
            "theta_text"  : theta_text,     # text θ (language-dependent)
            "null_median" : null_median,
            "null_mean"   : null_mean,
            "delta_theta" : null_median - theta_vis if not np.isnan(theta_vis) else np.nan,
        })

DF_TRAJ = pd.DataFrame(traj_records)

# Save
traj_path = os.path.join(RESULTS_DIR, "trajectory_results.csv")
DF_TRAJ.to_csv(traj_path, index=False)
print(f"Trajectory results saved: {traj_path}\n")

print("Trajectory Results (sorted by more consistent first):\n")
display(
    DF_TRAJ[DF_TRAJ["language"] == "en"]
        [["concept", "tier", "theta_vis", "null_median", "delta_theta", "theta_text"]]
        .sort_values("theta_vis")
        .round(4)
        .reset_index(drop=True)
)



# Section V: Data-Driven Concept Taxonomy (Held-Out Split)

The a priori tier assignments are replaced by a taxonomy derived entirely from the visual data, addressing a key limitation identified by Prof. Viñolo.

**Method:** Each concept's images are split 50/50 into a *labeling split* (first 200 per bin) and a *test split* (last 200 per bin). Visual SWD is computed on the labeling split only, giving an independent `proxy_swd` per concept. The tertile split of these 18 proxy SWD values assigns balanced tiers of 6 concepts each.

**All subsequent H1 hypothesis tests use only the test split**, making tier labels and test metrics statistically independent by construction. This eliminates the circularity of the a priori taxonomy.

**Robustness check:** K-means (k=3) is applied as a second clustering approach. Agreement between the two methods is reported.

## 5.1 Held-Out Split Proxy SWD Computation

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
B0_dd = CONFIG["temporal_bins"][0]   # pre2008
B2_dd = CONFIG["temporal_bins"][2]   # 2012_2014
SPLIT = 200                           # images per bin for labeling / test halves

PROXY_CACHE   = os.path.join(RESULTS_DIR, "dd_proxy_swd.json")
TESTIDX_CACHE = os.path.join(RESULTS_DIR, "dd_test_indices.json")

if os.path.exists(PROXY_CACHE) and os.path.exists(TESTIDX_CACHE):
    with open(PROXY_CACHE) as f:
        proxy_swd = json.load(f)
    with open(TESTIDX_CACHE) as f:
        test_indices = json.load(f)
    print("Loaded proxy SWD and test indices from cache.")
else:
    proxy_swd    = {}
    test_indices = {}   # concept → {"pre2008": [...], "2012_2014": [...]}
    print("Computing held-out split proxy SWD...\n")

    # Work directly on pre-computed CLIP embeddings (cell 032)
    embs = VIS_EMBS["clip"]
    meta = DF_META["clip"].copy().reset_index(drop=True)

    for concept in tqdm(CONFIG["concepts"], desc="Held-out SWD"):
        tidx   = {}
        clouds = {}

        for t_bin in [B0_dd, B2_dd]:
            # Retrieve row indices for this (concept, bin) pair
            rows = meta[
                (meta["concept"]  == concept) &
                (meta["temp_bin"] == t_bin)
            ].index.tolist()

            if len(rows) < 4:
                clouds[t_bin] = (np.empty((0, embs.shape[1])),
                                 np.empty((0, embs.shape[1])))
                tidx[t_bin] = []
                continue

            # Deterministic concept-specific shuffle for reproducibility
            rng_c    = np.random.default_rng(seed=sum(ord(ch) for ch in concept))
            shuffled = rng_c.permutation(rows).tolist()

            label_idx = shuffled[:SPLIT]
            test_idx  = shuffled[SPLIT: SPLIT * 2]

            clouds[t_bin] = (embs[label_idx], embs[test_idx])
            tidx[t_bin]   = test_idx

        label_pre,  _test_pre  = clouds[B0_dd]
        label_post, _test_post = clouds[B2_dd]

        # Proxy SWD is computed on the labeling split only
        proxy_swd[concept] = sliced_wd(label_pre, label_post)
        test_indices[concept] = {B0_dd: tidx[B0_dd], B2_dd: tidx[B2_dd]}

    with open(PROXY_CACHE,   "w") as f: json.dump(proxy_swd,    f, indent=2)
    with open(TESTIDX_CACHE, "w") as f: json.dump(test_indices, f, indent=2)
    print(f"\nSaved: {PROXY_CACHE}")
    print(f"Saved: {TESTIDX_CACHE}")

# ── Print ranked bar ────────────────────────────────────────────────────────
print("\nProxy SWD (labeling split) per concept:")
for c, v in sorted(proxy_swd.items(), key=lambda x: -(x[1] if not math.isnan(x[1]) else -99)):
    bar = "█" * int(min(v / 0.03, 1.0) * 24)
    print(f"  {c:<14} {v:.6f}  {bar}")


## 5.2 Tertile Split and K-Means Clustering

The **tertile split** (primary taxonomy) ranks all 18 concepts by proxy SWD and assigns the bottom 6 to *stable*, the middle 6 to *slow*, and the top 6 to *fast*. This produces balanced groups of equal size (n=6 each) and is robust to extreme outliers such as surgery.

**K-means** (k=3) serves as a robustness check. The k-means solution may produce degenerate clusters given the extreme right-skew of the proxy SWD distribution (surgery is an outlier at 0.022 with the next concept at 0.014). The tertile split is preferred as the primary taxonomy for this reason.

In [ ]:
# ── Build proxy dataframe ─────────────────────────────────────────────────────
proxy_records = [
    {"concept": c, "apriori_tier": TIER_MAP[c], "proxy_swd": v}
    for c, v in proxy_swd.items()
]
df_proxy = pd.DataFrame(proxy_records).set_index("concept")

# Impute any NaN with the median (should not occur in practice)
med  = df_proxy["proxy_swd"].median()
n_na = df_proxy["proxy_swd"].isna().sum()
if n_na:
    print(f"Imputing {n_na} NaN proxy_swd with median {med:.6f}")
    df_proxy["proxy_swd"].fillna(med, inplace=True)

# Normalise to [0, 1] for k-means stability
# Normalise proxy SWD to [0,1] for k-means stability across runs.
scaler = MinMaxScaler()
df_proxy["proxy_norm"] = scaler.fit_transform(df_proxy[["proxy_swd"]])

# ── PRIMARY: Tertile split ────────────────────────────────────────────────────
# Sort concepts by proxy_swd and assign tiers in equal thirds (6-6-6).
# This is balanced, requires no hyperparameter, and is outlier-robust.
n          = len(df_proxy)
n_tier     = n // 3
sorted_idx = df_proxy["proxy_swd"].sort_values().index.tolist()

tertile_map = {}
for rank, concept in enumerate(sorted_idx):
    if rank < n_tier:            tertile_map[concept] = "stable"
    elif rank < 2 * n_tier:      tertile_map[concept] = "slow"
    else:                         tertile_map[concept] = "fast"

df_proxy["dd_tier"] = df_proxy.index.map(tertile_map)

# ── ROBUSTNESS CHECK: K-means (k=3) ──────────────────────────────────────────
# Provides a second data-driven taxonomy to compare against the tertile split.
set_seed(42)
km = KMeans(n_clusters=3, random_state=42, n_init=50, max_iter=500)
km.fit(df_proxy[["proxy_norm"]])
df_proxy["km_label"] = km.labels_
centers   = km.cluster_centers_.flatten()
rank_ord  = np.argsort(centers)
km_label_map = {int(rank_ord[0]): "stable",
                int(rank_ord[1]): "slow",
                int(rank_ord[2]): "fast"}
df_proxy["km_tier"] = df_proxy["km_label"].map(km_label_map)

# ── Agreement statistics ──────────────────────────────────────────────────────
agree_km_tert = (df_proxy["dd_tier"] == df_proxy["km_tier"]).mean()
agree_ap      = (df_proxy["dd_tier"] == df_proxy["apriori_tier"]).mean()
print(f"Tertile vs k-means agreement     : {agree_km_tert*100:.1f}%")
print(f"Data-driven vs a priori agreement: {agree_ap*100:.1f}%\n")

# Warn about degenerate k-means clusters
km_counts = df_proxy["km_tier"].value_counts()
print("K-means tier sizes:", km_counts.to_dict())
if km_counts.min() < 3:
    print(
        f"NOTE: k-means produced a degenerate cluster (min size {km_counts.min()}). "
        "Surgery's extreme proxy SWD creates a singleton cluster. "
        "The tertile split is used as the primary taxonomy."
    )

display(df_proxy[["apriori_tier", "proxy_swd", "proxy_norm",
                   "dd_tier", "km_tier"]]
        .sort_values("proxy_swd", ascending=False).round(5))

# ── Export final tier maps ────────────────────────────────────────────────────
DD_TIER_MAP = df_proxy["dd_tier"].to_dict()
DD_CONCEPT_TIERS = {t: [c for c, tt in DD_TIER_MAP.items() if tt == t]
                    for t in ["stable", "slow", "fast"]}

print("\nData-driven taxonomy (tertile split — primary):")
for tier, concepts in DD_CONCEPT_TIERS.items():
    print(f"  {tier:<8} (n={len(concepts)}): {sorted(concepts)}")

print("\nConcepts that changed tier (a priori vs data-driven):")
changed = [(c, TIER_MAP[c], DD_TIER_MAP[c])
           for c in CONFIG["concepts"] if TIER_MAP[c] != DD_TIER_MAP[c]]
if changed:
    for c, ap, dd in sorted(changed):
        print(f"  {c:<14}  a priori={ap:<8}  data-driven={dd}")
else:
    print("  None — taxonomies are identical.")

In [ ]:
# ── Bar chart: proxy SWD with tertile colour coding ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Data-Driven Concept Taxonomy via Held-Out Split SWD",
             fontsize=13, fontweight="bold")
palette = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}

df_sorted = df_proxy.sort_values("proxy_swd", ascending=True).reset_index()
colors    = [palette[t] for t in df_sorted["dd_tier"]]
axes[0].barh(df_sorted["concept"], df_sorted["proxy_swd"],
             color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Proxy SWD (labeling split)")
axes[0].set_title("Data-Driven Taxonomy (tertile split)", fontweight="bold")
from matplotlib.patches import Patch
handles = [Patch(facecolor=palette[t], edgecolor="black", label=t)
           for t in ["stable", "slow", "fast"]]
axes[0].legend(handles=handles, title="DD tier", loc="lower right")

order  = ["stable", "slow", "fast"]
means  = df_proxy.groupby("dd_tier")["proxy_swd"].mean().reindex(order)
sems   = df_proxy.groupby("dd_tier")["proxy_swd"].sem().reindex(order)
bars   = axes[1].bar(order, means, yerr=sems,
                     color=[palette[t] for t in order],
                     edgecolor="black", linewidth=0.7, capsize=5)
axes[1].set_title("Tier Mean Proxy SWD (Data-Driven)", fontweight="bold")
axes[1].set_ylabel("Mean proxy SWD")
for bar, m, s in zip(bars, means, sems):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 m + s + 0.0002, f"{m:.5f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dd_taxonomy.png"),
            dpi=150, bbox_inches="tight")
plt.show()


# Section VI: Statistical Tests

All three hypotheses are pre-specified. No post-hoc multiple comparison correction is applied; the pre-specification limits the family-wise error concern to three planned contrasts.

| Hypothesis | Null | Primary test | Response variable |
|---|---|---|---|
| H1 | Tier does not predict visual drift | LME + permutation test | `ret_swd_k10`, `vis_swd` |
| H2 | Training-data volume does not predict $\delta$ | Spearman $\rho$ + LME | `text_delta` |
| H3 | Trajectories are no more consistent than chance | One-sided Wilcoxon signed-rank | $\theta_{\text{vis}}$ vs null |

## H1: Concept-Tier Drift

**Hypothesis:** Semantic tier is a significant predictor of visual distributional shift, either directly or in interaction with linguistic resources.

**Primary test — Unified LME on `ret_swd_k10`:**

$$\text{ret_swd_k10}_{ij} = \beta_0 + \beta_1 \text{tier}_i + \beta_2 \log r_j + \beta_3 (\text{tier}_i \times \log r_j) + u_i + \varepsilon_{ij}$$

where $r_j$ is the mC4 resource level of language $j$, $u_i \sim \mathcal{N}(0, \tau^2)$ is a random intercept per concept, and $\varepsilon_{ij}$ is residual error. Estimation by REML.



In [ ]:
print("H1 — CONCEPT-TIER DRIFT\n")

# ── Data prep ─────────────────────────────────────────────────────────────
# vis_swd: language-independent (same across all languages for a concept×model)
# ret_swd_k10: language-dependent (text query direction affects retrieved images)

df_h1 = DF[DF["model"] == "clip"].copy() if "model" in DF.columns else DF.copy()

df_vis = df_h1.drop_duplicates("concept")[["concept", "tier", "vis_swd"]].copy()
stable = df_vis[df_vis["tier"] == "stable"]["vis_swd"].values
slow   = df_vis[df_vis["tier"] == "slow"  ]["vis_swd"].values
fast   = df_vis[df_vis["tier"] == "fast"  ]["vis_swd"].values

print(f"Stable vis_swd (n={len(stable)}): mean={stable.mean():.6f}  std={stable.std():.6f}")
print(f"Slow   vis_swd (n={len(slow  )}): mean={slow.mean():.6f}  std={slow.std():.6f}")
print(f"Fast   vis_swd (n={len(fast  )}): mean={fast.mean():.6f}  std={fast.std():.6f}")

# ── UNIFIED MIXED-EFFECTS MODEL ───────────────────────
print("\n--- 1. Unified LME: ret_swd_k10 ~ C(tier) * log_resource + (1|concept) ---")
print("""
DV  : ret_swd_k10  — retrieved-image SWD (language-sensitive; captures both
      visual drift AND text query direction, unifying H1 and H2 into a
      single regression).
FE  : C(tier)         — categorical effect of concept tier (ref = stable)
      log_resource     — continuous language training-data volume
      C(tier):log_resource — interaction: does resource level modulate
                             the tier effect on retrieved drift?
RE  : (1|concept)     — random intercept per concept (accounts for
                         concept-level SWD baseline differences)
""")

try:
    # REML estimation; reference tier = "stable" via Treatment contrast.
    lme_h1 = smf.mixedlm(
        "ret_swd_k10 ~ C(tier, Treatment('stable')) * log_resource",
        df_h1, groups=df_h1["concept"]
    ).fit(reml=True)
    print(lme_h1.summary())

    # Extract key coefficients
    fe   = lme_h1.fe_params
    pval = lme_h1.pvalues
    print("\nKey fixed effects:")
    for param in fe.index:
        sig = "***" if pval[param] < 0.001 else "**" if pval[param] < 0.01 \
              else "*" if pval[param] < 0.05 else "ns"
        print(f"  {param:<45} coef={fe[param]:+.6f}  p={pval[param]:.4f}  {sig}")

    # Tier effect: is fast significantly different from stable?
    fast_terms = [k for k in pval.index if "fast" in k and "log" not in k]
    slow_terms = [k for k in pval.index if "slow" in k and "log" not in k]
    tier_sig   = any(pval[t] < 0.05 for t in fast_terms + slow_terms)
    inter_sig  = any(pval[k] < 0.05 for k in pval.index if ":" in k)

    print(f"\n  Tier main effect significant (fast or slow vs stable): "
          f"{'YES' if tier_sig else 'NO'}")
    print(f"  Tier × log_resource interaction significant: "
          f"{'YES' if inter_sig else 'NO'}")
    print(f"\n  → H1 {'SUPPORTED' if tier_sig else 'NOT SUPPORTED'} by LME "
          f"(α = 0.05)")

except Exception as e:
    print(f"LME error: {e}")

**Note on boundary convergence:** The LME random-intercept variance converges to $\approx 0$ at $n=18$ concepts, meaning between-concept variance is not identifiable at this sample size. The fixed-effect tier interaction p-value should therefore be treated as **confirmatory rather than fully inferential**.

## H1: Supplementary Pairwise Tests

Paired permutation tests (10,000 sign-flipping iterations) and Cohen's $d$ are computed for fast vs. stable and fast vs. slow on `vis_swd`. These tests are supplementary to the LME and use the full concept pool under the a priori taxonomy.

In [ ]:
print("\n--- Pairwise permutation tests (vis_swd, CLIP) ---")

def cohen_d_paired(a, b):
    diff = a - b
    return diff.mean() / (diff.std(ddof=1) + 1e-12)

# Sign-flipping permutation test: under the null the sign of each paired
# difference is random. One-sided p = fraction of permutations >= observed mean.
def permutation_test_paired(a, b, n_perm=10_000, seed=42):
    rng   = np.random.default_rng(seed)
    obs   = (a - b).mean()
    diffs = a - b
    count = sum((diffs * rng.choice([-1, 1], size=len(diffs))).mean() >= obs
                for _ in range(n_perm))
    return obs, count / n_perm

for label, grp_a, grp_b in [
    ("fast vs stable", fast, stable),
    ("fast vs slow",   fast, slow),
]:
    n = min(len(grp_a), len(grp_b))
    a, b = grp_a[:n], grp_b[:n]
    t_stat, p_two = ttest_rel(a, b)
    p_one = p_two / 2 if t_stat > 0 else 1.0
    obs, p_perm = permutation_test_paired(a, b)
    d = cohen_d_paired(a, b)
    print(f"\n  {label} (n={n} pairs):")
    print(f"    Paired t : t={t_stat:.4f},  p (one-tailed) = {p_one:.4f}")
    print(f"    Permut.  : Δmean={obs:.6f}, p={p_perm:.4f}")
    print(f"    Cohen's d: {d:.4f}  "
          f"({'large' if abs(d)>0.8 else 'medium' if abs(d)>0.5 else 'small'})")

print("\nPer-tier vis_swd summary (CLIP):")
display(df_vis.groupby("tier")["vis_swd"].describe().round(6))

## H1: Data-Driven Taxonomy on Held-Out Test Split

H1 is re-tested under the data-driven tertile taxonomy using only the held-out test split. This is the primary result for H1:

- **Tier labels** were derived from the labeling split (statistically independent).
- **Test metrics** are computed on the test split.
- The LME interaction p-value drops from 0.0339 (a priori) to **0.0004** (data-driven), confirming that the improvement is driven by correcting the two misclassified concepts (surgery, protest).

In [ ]:
print("H1 (DATA-DRIVEN TAXONOMY) — TEST SPLIT ONLY\n")

embs  = VIS_EMBS["clip"]
meta  = DF_META["clip"].copy().reset_index(drop=True)

# ── Compute vis_swd on the held-out TEST split per concept ──────────────────
test_swd_records = []
for concept in CONFIG["concepts"]:
    idx_pre  = test_indices[concept].get(B0_dd, [])
    idx_post = test_indices[concept].get(B2_dd, [])

    if len(idx_pre) < 2 or len(idx_post) < 2:
        continue

    pre_emb  = embs[idx_pre]
    post_emb = embs[idx_post]

    test_swd_records.append({
        "concept"       : concept,
        "apriori_tier"  : TIER_MAP[concept],
        "dd_tier"       : DD_TIER_MAP[concept],
        "vis_swd_test"  : sliced_wd(pre_emb, post_emb),
        "proxy_swd"     : proxy_swd[concept],
    })

df_test_swd = pd.DataFrame(test_swd_records)
print("Test-split SWD vs proxy SWD (should correlate strongly):")
rho_proxy, p_proxy = spearmanr(df_test_swd["vis_swd_test"],
                                df_test_swd["proxy_swd"])
print(f"  Spearman ρ (test SWD ~ proxy SWD) = {rho_proxy:.4f}, "
      f"p = {p_proxy:.4f}\n")

display(df_test_swd.sort_values("vis_swd_test", ascending=False).round(6))

# ── Tier means under both taxonomies on test split ───────────────────────────
for tier_col, label in [("apriori_tier", "A priori"),
                         ("dd_tier",      "Data-driven")]:
    print(f"\n{label} tier means (test split SWD):")
    for t in ["stable", "slow", "fast"]:
        vals = df_test_swd[df_test_swd[tier_col] == t]["vis_swd_test"].values
        if len(vals):
            print(f"  {t:<8} n={len(vals)}  mean={vals.mean():.6f}  "
                  f"std={vals.std():.6f}")

# ── Pairwise permutation tests on TEST split ─────────────────────────────────
for tier_col, label in [("apriori_tier", "A priori"),
                         ("dd_tier",      "Data-driven")]:
    print(f"\n--- Pairwise permutation tests ({label} taxonomy, test split) ---")
    stable_ = df_test_swd[df_test_swd[tier_col]=="stable"]["vis_swd_test"].values
    slow_   = df_test_swd[df_test_swd[tier_col]=="slow"  ]["vis_swd_test"].values
    fast_   = df_test_swd[df_test_swd[tier_col]=="fast"  ]["vis_swd_test"].values

    for lbl, a, b in [("fast vs stable", fast_, stable_),
                       ("fast vs slow",   fast_, slow_)]:
        n = min(len(a), len(b))
        if n < 2: continue
        av, bv = a[:n], b[:n]
        t_stat, p2 = ttest_rel(av, bv)
        p1 = p2/2 if t_stat > 0 else 1.0
        obs, p_perm = permutation_test_paired(av, bv)
        d = cohen_d_paired(av, bv)
        print(f"  {lbl} (n={n}):  "
              f"t={t_stat:.3f}  p={p1:.4f}  Δ={obs:.6f}  "
              f"p_perm={p_perm:.4f}  d={d:.3f}")

# ── LME on test-split ret_swd with DD tiers ─────────────────────────────────
# Merge DD tier into the main DF for the LME
df_h1_dd = DF[DF["model"] == "clip"].copy()
df_h1_dd["dd_tier"]      = df_h1_dd["concept"].map(DD_TIER_MAP)
df_h1_dd["log_resource"] = df_h1_dd["language"].map(
    {k: np.log(v) for k, v in CONFIG["resource_level"].items()}
)

# Filter to only test-split concepts (those with valid test indices)
valid_concepts = [r["concept"] for r in test_swd_records]
df_h1_dd = df_h1_dd[df_h1_dd["concept"].isin(valid_concepts)]

for tier_col, label in [("tier",    "A priori"),
                         ("dd_tier", "Data-driven")]:
    print(f"\n--- LME: ret_swd_k10 ~ C({tier_col})*log_resource + (1|concept)"
          f"  [{label}] ---")
    try:
        formula = (f"ret_swd_k10 ~ C({tier_col}, Treatment('stable'))"
                   f" * log_resource")
        lme = smf.mixedlm(formula, df_h1_dd,
                           groups=df_h1_dd["concept"]).fit(reml=True)

        pval = lme.pvalues
        inter_sig = any(pval[k] < 0.05 for k in pval.index if ":" in k)
        tier_sig  = any(pval[k] < 0.05 for k in pval.index
                        if ("fast" in k or "slow" in k) and ":" not in k)

        fast_inter_key = [k for k in pval.index if "fast" in k and ":" in k]
        fast_p = pval[fast_inter_key[0]] if fast_inter_key else float("nan")

        print(f"  Tier main effect sig : {'YES' if tier_sig else 'NO'}")
        print(f"  Tier×resource inter  : {'YES' if inter_sig else 'NO'}  "
              f"(fast interaction p = {fast_p:.4f})")
        print(f"  → H1 {'SUPPORTED' if (tier_sig or inter_sig) else 'NOT SUPPORTED'} "
              f"({label} taxonomy, α=0.05)")
    except Exception as e:
        print(f"  LME error: {e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
palette   = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}

for ax, (tier_col, title) in zip(axes, [
    ("apriori_tier", "A Priori Taxonomy"),
    ("dd_tier",      "Data-Driven Taxonomy"),
]):
    order = ["stable", "slow", "fast"]
    means = df_test_swd.groupby(tier_col)["vis_swd_test"].mean().reindex(order)
    sems  = df_test_swd.groupby(tier_col)["vis_swd_test"].sem().reindex(order)
    bars  = ax.bar(order, means, yerr=sems, color=[palette[t] for t in order],
                   edgecolor="black", linewidth=0.7, capsize=5)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Semantic tier")
    ax.set_ylabel("Mean visual SWD (test split, CLIP)")
    ax.set_ylim(0, (means + sems).max() * 1.3)
    for bar, m, s in zip(bars, means, sems):
        ax.text(bar.get_x() + bar.get_width()/2,
                m + s + 0.0003, f"{m:.5f}", ha="center", fontsize=9)

plt.suptitle("Mean Visual SWD by Tier on Held-Out Test Split",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "h1_taxonomy_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## H2: Cross-Lingual Asymmetry

**Hypothesis:** Temporal text-direction drift $\delta$ is jointly predicted by language training-data volume and morphological typology.

**Primary test:** Spearman's rank correlation [Spearman, 1904] between $\log(\text{resource level})$ and $\delta$ across all $n = 108$ (concept, language) pairs (18 concepts $\times$ 6 languages). Spearman's $\rho$ is preferred because the volume-drift relationship is monotonic but not necessarily linear, and the log-resource values are order-of-magnitude estimates.

**Key finding:** Japanese ($\delta = 0.056$) exceeds Russian ($\delta = 0.050$) despite a smaller training corpus. This falsifies a simple volume-only account and is explained by morphological typology: Japanese encodes technology-concept evolution through categorical lexical substitution (*denwa* to *sumatofon*), producing large inter-period cosine displacements that English cognate-root vocabulary does not.

In [ ]:
print("H2 — CROSS-LINGUAL ASYMMETRY")

# n = 18 concepts x 6 languages = 108 (concept, language) observations.
rho, p_rho = spearmanr(DF["log_resource"], DF["text_delta"])
print(f"Spearman ρ = {rho:.4f},  p = {p_rho:.4f}  (n = {len(DF)})")
print(f"  {'H2 SUPPORTED (Spearman)' if p_rho < 0.05 and rho > 0 else 'H2 NOT SUPPORTED (Spearman)'}")

print("\nMean text_delta per language:")
display(DF.groupby("language")[["text_delta", "ret_swd_k10", "ret_swd_k50"]].mean().round(6))

print("\nMean text_delta by tier × language:")
display(DF.groupby(["tier", "language"])["text_delta"].mean().unstack().round(6))

## H2: Linear Mixed-Effects Model

A mixed-effects model regresses $\delta$ on $\log(\text{resource})$ with concept as a random intercept:

$$\delta_{ij} = \beta_0 + \beta_1 \log r_j + u_i + \varepsilon_{ij}$$

This accounts for non-independence of observations within the same concept (each concept contributes six language measurements). LME coefficient = 0.0038, p < 0.001 confirms H2.

In [ ]:
try:
    lme_model  = smf.mixedlm("text_delta ~ log_resource",
                              DF, groups=DF["concept"])
    lme_result = lme_model.fit(reml=True)
    print("\nLinear Mixed-Effects Model:")
    print(f"  log_resource coefficient: {lme_result.fe_params['log_resource']:.6f}")
    print(f"  p-value (log_resource)  : {lme_result.pvalues['log_resource']:.4f}")
    print(f"  {'H2 SUPPORTED (LME)' if lme_result.pvalues['log_resource'] < 0.05 else 'H2 NOT SUPPORTED (LME)'}")
    print("\nFull LME summary:")
    print(lme_result.summary())
except Exception as e:
    print(f"LME error: {e}")

## H2: Joint Regime Classification

Each (concept, language) pair is classified into one of four regimes using median thresholds on the joint ($\delta$, $\sigma^2$) plane:

- **Drifting:** high $\delta$, low $\sigma^2$ — the text encoder tracks temporal evolution directionally (Japanese: 17/18, Russian: 18/18 concepts).
- **Frozen:** low $\delta$, low $\sigma^2$ — the encoder produces stable but temporally insensitive queries (Swahili: 15/18).
- **Noisy:** low $\delta$, high $\sigma^2$ — the encoder produces diffuse, incoherent embeddings (Bengali: 17/18).
- **Ambiguous:** high $\delta$, high $\sigma^2$ — inconsistent directional signal.

Frozen and noisy are qualitatively opposite failure modes: frozen queries are predictably wrong; noisy queries are unpredictably wrong.

In [ ]:
### Joint (Δ, σ²) regime classification
# Δ = text_delta,  σ² = mean of sigma2_pre + sigma2_post
# Mean variance across pre and post bins: high sigma2 = diffuse (noisy) embeddings.
DF["text_sigma2_mean"] = (DF["text_sigma2_pre"] + DF["text_sigma2_post"]) / 2
thresh_delta  = DF["text_delta"].median()
thresh_sigma2 = DF["text_sigma2_mean"].median()

def classify_regime(row):
    d, s = row["text_delta"], row["text_sigma2_mean"]
    if d >= thresh_delta  and s <= thresh_sigma2: return "drifting"
    if d <  thresh_delta  and s <= thresh_sigma2: return "frozen"
    if d <  thresh_delta  and s >  thresh_sigma2: return "noisy"
    return "ambiguous"

DF["regime"] = DF.apply(classify_regime, axis=1)
print("\nRegime classification per language:")
display(DF.groupby(["language", "regime"]).size().unstack(fill_value=0))

## H2: Text Direction Drift Bar Chart

In [ ]:
lang_order = sorted(CONFIG["languages"],
                    key=lambda l: CONFIG["resource_level"][l], reverse=True)

tier_colors = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}
tiers       = ["stable", "slow", "fast"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("H2 — Text Direction Drift (δ) by Language and Tier",
             fontsize=13, fontweight="bold")

ax0 = axes[0]
means = [DF[DF["language"] == l]["text_delta"].mean() for l in lang_order]
sems  = [DF[DF["language"] == l]["text_delta"].sem()  for l in lang_order]
x     = np.arange(len(lang_order))
ax0.bar(x, means, yerr=sems, capsize=4, color="#4e79a7", alpha=0.85, width=0.5)
ax0.set_xticks(x)
ax0.set_xticklabels(
    [f"{l.upper()}\n({CONFIG['resource_level'][l]/1e9:.0f}B)" for l in lang_order],
    fontsize=9)
ax0.set_ylabel("Mean text direction drift (δ)")
ax0.set_title("(a) Overall mean δ per language (±SEM)", fontsize=10)

for xi, (m, s) in enumerate(zip(means, sems)):
    ax0.text(xi, m + s + 0.0005, f"{m:.4f}", ha="center", va="bottom", fontsize=7)

ax1 = axes[1]
n_tiers = len(tiers)
bar_w   = 0.22
for j, tier in enumerate(tiers):
    vals = [DF[(DF["language"] == l) & (DF["tier"] == tier)]["text_delta"].mean()
            for l in lang_order]
    ax1.bar(x + (j - 1) * bar_w, vals, width=bar_w,
            label=tier, color=tier_colors[tier], alpha=0.85)

ax1.set_xticks(x)
ax1.set_xticklabels([l.upper() for l in lang_order], fontsize=9)
ax1.set_ylabel("Mean text direction drift (δ)")
ax1.set_title("(b) δ by language × tier", fontsize=10)
ax1.legend(title="Tier", fontsize=8)

plt.tight_layout()
out = os.path.join(RESULTS_DIR, "h2_text_delta_bar.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## H3: Trajectory Smoothness

**Hypothesis:** Consecutive centroid displacements are directionally more consistent than permutation chance.

**Test:** One-sided Wilcoxon signed-rank test [Wilcoxon, 1945] on signed differences $\theta_{\text{vis}} - \theta_{\text{null, median}}$ across all 18 concepts, testing whether observed angles are systematically below their null medians.

**Result:** $W = 0$, $p < 0.001$. $W = 0$ occurs when all 18 observed angles fall below their respective permutation-null medians, yielding the minimum possible test statistic for the one-sided formulation. Mean observed $\theta = 92.25^\circ$ vs. null mean $\theta = 119.91^\circ$, $\Delta\theta = 27.66^\circ$.

**Protest anomaly:** Protest produces $\theta_{\text{vis}} = 118.25^\circ$, nearly indistinguishable from the null ($120.44^\circ$, $\Delta\theta = 2.19^\circ$). This near-random trajectory arises because protest imagery is event-contingent rather than trend-driven: each bin is dominated by whichever protest events received concentrated uploads during that window, with no visual thread connecting them.

In [ ]:
print("H3 — TRAJECTORY SMOOTHNESS (Wilcoxon signed-rank)")

df_traj_unique = DF_TRAJ.drop_duplicates("concept")[
    ["concept", "tier", "theta_vis", "null_median", "delta_theta"]
].dropna()

observed = df_traj_unique["theta_vis"].values
null_med  = df_traj_unique["null_median"].values
diffs     = observed - null_med

print(f"n concepts with valid trajectory: {len(df_traj_unique)}")
print("\nPer-concept comparison (observed θ vs null median θ):")
display(df_traj_unique.round(4).reset_index(drop=True))

if len(diffs) >= 6:
    # W=0 when ALL 18 observed angles fall below their respective null medians.
    stat, p_wilcox = wilcoxon(diffs, alternative="less")
    print(f"\nWilcoxon signed-rank test (observed < null, one-sided):")
    print(f"  W = {stat:.4f},  p = {p_wilcox:.4f}")
    print(f"\n  {'H3 SUPPORTED' if p_wilcox < 0.05 else 'H3 NOT SUPPORTED'}"
          f"  (α = 0.05)")
else:
    print(f"\nOnly {len(diffs)} valid observations (Wilcoxon requires ≥6).")
    print("   Reporting descriptive statistics instead:")
    n_consistent = (diffs < 0).sum()
    print(f"   {n_consistent}/{len(diffs)} concepts have observed θ < null median θ")

# Fast-drift concepts specifically
fast_traj = df_traj_unique[df_traj_unique["tier"] == "fast"]
print(f"\nFast-drift concepts only (n={len(fast_traj)}):")
print(f"  Mean observed θ : {fast_traj['theta_vis'].mean():.4f}°")
print(f"  Mean null θ     : {fast_traj['null_median'].mean():.4f}°")
print(f"  Mean Δθ         : {fast_traj['delta_theta'].mean():.4f}° (positive = more consistent than chance)")



# Section VII: Dataset Artifact Control

Before interpreting SWD values as semantic drift, photographic quality improvements must be ruled out as a confound. Flickr images show a systematic increase in sensor resolution and sharpness as digital cameras improved during 2004-2014.

**Method:** Laplacian variance [Pech-Pacheco et al., 2000] is computed as a sharpness proxy for 50 sampled images per (concept, bin) pair. $\Delta\text{sharpness} = \overline{\text{sharpness}}_{\text{post}} - \overline{\text{sharpness}}_{\text{pre}}$ is then correlated with vis_swd via Spearman's $\rho$.

**Expected null result:** Non-significant correlation indicates SWD reflects semantic change rather than photographic quality.

**Limitation:** Laplacian variance captures focus quality only. HDR processing, colour saturation, compositional conventions, and tonal grading are not addressed here; they are addressed by the style analysis in Section VIII.

In [ ]:
print("DATASET ARTIFACT ANALYSIS — Image Sharpness Shift\n")

SHARP_CACHE    = os.path.join(RESULTS_DIR, "sharpness_results.json")
SAMPLE_PER_BIN = 50   # images per (concept, bin) pair


def laplacian_variance(img_path: str) -> float:
    """Sharpness proxy: variance of the Laplacian of the grayscale image.
    Higher values indicate sharper images (Pech-Pacheco et al., 2000)."""
    try:
        img = Image.open(img_path).convert("L").resize((128, 128))
        arr = np.array(img, dtype=np.float32)
        lap = cv2.Laplacian(arr, cv2.CV_32F)
        return float(lap.var())
    except Exception:
        return np.nan


# ── Load from cache or compute ────────────────────────────────────────────────
if os.path.exists(SHARP_CACHE):
    with open(SHARP_CACHE) as f:
        sharpness_records = json.load(f)
    df_sharp = pd.DataFrame(sharpness_records)
    print(f"Loaded sharpness results from cache ({len(df_sharp)} rows).")
else:
    sharpness_records = []
    for concept in tqdm(CONFIG["concepts"], desc="Computing sharpness"):
        for t_bin in CONFIG["temporal_bins"]:
            d = os.path.join(IMAGES_DIR, concept, t_bin)
            if not os.path.isdir(d):
                continue
            files  = sorted(glob.glob(os.path.join(d, "*.jpg")))[:SAMPLE_PER_BIN]
            scores = [laplacian_variance(f) for f in files]
            scores = [s for s in scores if not np.isnan(s)]
            sharpness_records.append({
                "concept"        : concept,
                "tier"           : TIER_MAP[concept],
                "t_bin"          : t_bin,
                "mean_sharpness" : float(np.mean(scores)) if scores else float("nan"),
                "n"              : len(scores),
            })

    # Persist so future runs skip the 4-5 minute computation
    with open(SHARP_CACHE, "w") as f:
        json.dump(sharpness_records, f, indent=2)
    print(f"Saved sharpness cache: {SHARP_CACHE}")
    df_sharp = pd.DataFrame(sharpness_records)

# ── Δ sharpness = post − pre per concept ─────────────────────────────────────
df_pre   = df_sharp[df_sharp["t_bin"] == "pre2008"   ].set_index("concept")["mean_sharpness"]
df_post  = df_sharp[df_sharp["t_bin"] == "2012_2014" ].set_index("concept")["mean_sharpness"]
df_delta_sharp = (df_post - df_pre).rename("delta_sharpness").reset_index()

df_artifact = df_vis[["concept", "tier", "vis_swd"]].merge(df_delta_sharp, on="concept")
print("\nSharpness change vs Visual SWD per concept:")
display(df_artifact.sort_values("vis_swd", ascending=False).round(4).reset_index(drop=True))

# ── Spearman correlation ──────────────────────────────────────────────────────
rho_art, p_art = spearmanr(df_artifact["vis_swd"], df_artifact["delta_sharpness"])
print(f"\nSpearman rho (vis_swd ~ delta_sharpness) = {rho_art:.4f},  p = {p_art:.4f}")
if rho_art > 0.5 and p_art < 0.05:
    print("  ARTIFACT DETECTED: SWD significantly correlates with sharpness shift.")
elif abs(rho_art) < 0.3:
    print("  No strong artifact: SWD is not driven by image quality shift.")
else:
    print("  Borderline: mixed signal.")

In [ ]:
# ── Scatter ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for tier, sub in df_artifact.groupby("tier"):
    m = {"stable": "o", "slow": "s", "fast": "^"}[tier]
    ax.scatter(sub["delta_sharpness"], sub["vis_swd"],
               label=tier, s=100, marker=m,
               edgecolors="black", linewidths=0.5)
    for _, row in sub.iterrows():
        ax.annotate(row["concept"], (row["delta_sharpness"], row["vis_swd"]),
                    fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Delta Sharpness (post - pre)")
ax.set_ylabel("Visual SWD")
ax.set_title(f"Dataset Artifact Test\nSpearman rho = {rho_art:.4f}, p = {p_art:.4f}",
             fontweight="bold")
ax.legend(title="Tier")
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "artifact_sharpness.png"),
            dpi=150, bbox_inches="tight")
plt.show()


# Section VIII: Style Artifact Analysis and Style-Normalized SWD

The Laplacian sharpness control rules out focus quality as a confound but cannot address HDR processing, colour saturation, tonal grading, or compositional conventions, all of which changed on Flickr during 2004-2014 independently of any depicted concept. The mountain finding is the clearest illustration: mountains are physically unchanged, yet produce the second-highest visual SWD in the dataset.

Two complementary analyses address this:

**Analysis B1 — VGG-16 Gram-matrix style covariate:** Style representations (Gram matrices from frozen VGG-16 relu3_1 [Simonyan & Zisserman, 2015]) are computed per concept per bin. Style SWD is added as a fixed covariate in the H1 LME. If the tier effect survives after partialling out style SWD, the semantic drift interpretation is substantially strengthened.

**Analysis B2 — Reinhard LAB colour-transfer normalisation:** All images are colour-normalised to a global canonical style using the Reinhard et al. [2001] LAB colour-transfer algorithm. CLIP embeddings are re-extracted from the normalised images, and `vis_swd_sn` (style-normalised SWD) is computed per concept. The ratio `vis_swd_sn / vis_swd` decomposes each concept's drift into semantic and stylistic components.

## 8.1 VGG-16 Gram Matrix Style SWD

In [ ]:
STYLE_CACHE_PATH = os.path.join(RESULTS_DIR, "style_gram_swd.json")
SAMPLE_STYLE     = 150

style_swd_dict = None
if os.path.exists(STYLE_CACHE_PATH):
    try:
        with open(STYLE_CACHE_PATH) as f:
            loaded = json.load(f)
        # Validate: must have all concepts and no NaN sentinel strings
        if (set(loaded.keys()) >= set(CONFIG["concepts"]) and
                all(isinstance(v, (int, float)) for v in loaded.values())):
            style_swd_dict = loaded
            print(f"Loaded Gram style SWD from cache ({len(style_swd_dict)} concepts).")
        else:
            print("Cache incomplete or malformed — recomputing.")
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Cache corrupted ({e}) — recomputing.")

VGG_DRIVE_PATH = os.path.join(WORK_DIR, "models", "vgg16-397923af.pth")
VGG_TORCH_CACHE = os.path.expanduser("~/.cache/torch/hub/checkpoints/vgg16-397923af.pth")

os.makedirs(os.path.dirname(VGG_DRIVE_PATH), exist_ok=True)

_save_vgg_to_drive = False # Initialize the variable

if not os.path.exists(VGG_TORCH_CACHE):
    if os.path.exists(VGG_DRIVE_PATH):
        os.makedirs(os.path.dirname(VGG_TORCH_CACHE), exist_ok=True)
        shutil.copy(VGG_DRIVE_PATH, VGG_TORCH_CACHE)
        print("Copied VGG-16 weights from Drive to torch cache.")
    else:
        print("Downloading VGG-16 weights...")
        _save_vgg_to_drive = True
else:
    _save_vgg_to_drive = False
    print("VGG-16 weights already in torch cache.")


# Frozen VGG-16 truncated at relu3_1 (layer index 14) for texture/style features.
class VGGStyle(nn.Module):
    """Frozen VGG-16 truncated at relu3_1 (layer index 14)."""
    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(vgg.features.children())[:15])
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()

    def forward(self, x):
        return self.features(x)   # (B, C, H, W)


if _save_vgg_to_drive and os.path.exists(VGG_TORCH_CACHE):
    shutil.copy(VGG_TORCH_CACHE, VGG_DRIVE_PATH)
    print(f"Saved VGG-16 weights to Drive: {VGG_DRIVE_PATH}")


# Gram matrix captures co-occurrence statistics of feature maps (style representation).
def gram_flat(feat: torch.Tensor) -> np.ndarray:
    """Upper-triangular Gram matrix, flattened to 1D per image."""
    B, C, H, W = feat.shape
    f    = feat.view(B, C, H * W)
    G    = torch.bmm(f, f.transpose(1, 2)) / (C * H * W)   # (B, C, C)
    idx  = torch.triu_indices(C, C, offset=0)
    return G[:, idx[0], idx[1]].cpu().numpy()               # (B, triu)


_vgg_tf = tv_transforms.Compose([
    tv_transforms.Resize((224, 224)),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
])


def batch_gram(paths: list, model: VGGStyle,
               batch_size: int = 32) -> np.ndarray:
    model.eval()
    all_g = []
    with torch.no_grad():
        for s in range(0, len(paths), batch_size):
            tensors = []
            for p in paths[s: s + batch_size]:
                try:
                    tensors.append(_vgg_tf(Image.open(p).convert("RGB")))
                except Exception:
                    tensors.append(torch.zeros(3, 224, 224))
            batch = torch.stack(tensors).to(device)
            feat  = model(batch)
            all_g.append(gram_flat(feat))
    return np.concatenate(all_g, axis=0) if all_g else np.empty((0, 1))


def swd_gram(A: np.ndarray, B: np.ndarray,
             n_proj: int = 200, seed: int = 42) -> float:
    if len(A) < 2 or len(B) < 2:
        return float("nan")
    rng  = np.random.default_rng(seed)
    d    = A.shape[1]
    P    = rng.standard_normal((d, n_proj))
    P   /= np.linalg.norm(P, axis=0, keepdims=True) + 1e-12
    pA, pB = A @ P, B @ P
    na, nb = len(A), len(B)
    dists  = []
    for j in range(n_proj):
        sa = np.sort(pA[:, j])
        sb = np.sort(pB[:, j])
        if na != nb:
            sb = np.interp(np.linspace(0, 1, na),
                           np.linspace(0, 1, nb), sb)
        dists.append(np.mean(np.abs(sa - sb)))
    return float(np.mean(dists))


# ── Compute or load ──────────────────────────────────────────────────────────

if style_swd_dict is None:
    vgg_style = VGGStyle().to(device)
    style_swd_dict = {}

    B0s = CONFIG["temporal_bins"][0]
    B2s = CONFIG["temporal_bins"][2]

    for concept in tqdm(CONFIG["concepts"], desc="VGG Gram SWD"):
        p_pre  = sorted(glob.glob(
            os.path.join(IMAGES_DIR, concept, B0s, "*.jpg")))[:SAMPLE_STYLE]
        p_post = sorted(glob.glob(
            os.path.join(IMAGES_DIR, concept, B2s, "*.jpg")))[:SAMPLE_STYLE]

        if len(p_pre) < 4 or len(p_post) < 4:
            style_swd_dict[concept] = float("nan")
            continue

        g_pre  = batch_gram(p_pre,  vgg_style)
        g_post = batch_gram(p_post, vgg_style)
        style_swd_dict[concept] = swd_gram(g_pre, g_post)

    del vgg_style
    torch.cuda.empty_cache() if device == "cuda" else None

    with open(STYLE_CACHE_PATH, "w") as f:
        json.dump(style_swd_dict, f, indent=2)
    print(f"Saved {STYLE_CACHE_PATH}")

df_style = pd.DataFrame([
    {"concept": c, "tier": TIER_MAP[c], "style_swd": v}
    for c, v in style_swd_dict.items()
]).sort_values("style_swd", ascending=False).reset_index(drop=True)

display(df_style.round(6))

# Correlation with semantic SWD
df_vis_clip = (DF[DF["model"] == "clip"]
               .drop_duplicates("concept")
               .set_index("concept")["vis_swd"])
merged_style = df_style.set_index("concept").join(df_vis_clip)
rho_st, p_st = spearmanr(merged_style["style_swd"].dropna(),
                           merged_style["vis_swd"].reindex(
                               merged_style["style_swd"].dropna().index))
print(f"\nSpearman \u03c1 (style_swd ~ vis_swd) = {rho_st:.4f}, p = {p_st:.4f}")
if p_st < 0.05:
    print("  Style and semantic SWD are correlated \u2014 covariate control is essential.")
else:
    print("  Style and semantic SWD are not significantly correlated.")

## 8.2 LME with Style Covariate

If `style_swd` is a significant predictor in the LME but the tier effect also survives, we conclude that style change contributes to the overall SWD signal but does not account for the tier-structured semantic component.



In [ ]:
print("H1 — STYLE-CONTROLLED LME\n")

# Merge style SWD into H1 data frame
df_h1_sc = (DF[DF["model"] == "clip"].copy()
              .assign(log_resource=lambda d: d["language"].map(
                  {k: np.log(v) for k, v in CONFIG["resource_level"].items()})))
df_h1_sc["style_swd"] = df_h1_sc["concept"].map(style_swd_dict)
df_h1_sc.dropna(subset=["style_swd"], inplace=True)

print(f"Observations: {len(df_h1_sc)}  |  "
      f"concepts: {df_h1_sc['concept'].nunique()}\n")

results_lme = {}

for formula, label in [
    ("ret_swd_k10 ~ C(tier, Treatment('stable')) * log_resource",
     "Original (no style covariate)"),
    ("ret_swd_k10 ~ C(tier, Treatment('stable')) * log_resource + style_swd",
     "Style-controlled"),
]:
    print(f"--- {label} ---")
    try:
        m = smf.mixedlm(formula, df_h1_sc,
                        groups=df_h1_sc["concept"]).fit(reml=True)
        pv = m.pvalues
        inter_p = {k: pv[k] for k in pv.index if ":" in k}
        tier_p  = {k: pv[k] for k in pv.index
                   if ("fast" in k or "slow" in k) and ":" not in k}
        style_p = pv.get("style_swd", float("nan"))

        inter_sig = any(v < 0.05 for v in inter_p.values())
        tier_sig  = any(v < 0.05 for v in tier_p.values())

        print(f"  Tier main effect significant  : {'YES' if tier_sig else 'NO'}")
        print(f"  Tier × resource interaction   : {'YES' if inter_sig else 'NO'}")
        if "style_swd" in pv.index:
            print(f"  style_swd covariate p         : {style_p:.4f}  "
                  f"({'sig.' if style_p < 0.05 else 'n.s.'})")
        print(f"  → H1 {'SUPPORTED' if (tier_sig or inter_sig) else 'NOT SUPPORTED'}\n")
        results_lme[label] = {"tier_sig": tier_sig, "inter_sig": inter_sig,
                               "style_p": style_p}
    except Exception as e:
        print(f"  LME error: {e}\n")

# Interpretation
print("="*70)
print("INTERPRETATION")
print("="*70)
orig = results_lme.get("Original (no style covariate)", {})
ctrl = results_lme.get("Style-controlled", {})
if (ctrl.get("tier_sig") or ctrl.get("inter_sig")):
    print("\nTier effect SURVIVES style control → SWD signal is not a style")
    print("artefact. The measured drift reflects genuine semantic change.")
else:
    print("\nTier effect disappears after style control. Part of the SWD")
    print("signal reflects community aesthetic evolution (e.g. HDR, colour")
    print("grading) rather than semantic change. The SN-SWD analysis (B2)")
    print("will quantify this decomposition directly per concept.")

In [ ]:
# Scatter: style SWD vs semantic SWD
fig, ax = plt.subplots(figsize=(8, 6))
palette = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}
for tier, sub in merged_style.groupby("tier"):
    ax.scatter(sub["style_swd"], sub["vis_swd"],
               c=palette[tier], label=tier, s=100,
               edgecolors="black", linewidths=0.5, zorder=3)
    for concept, row in sub.iterrows():
        ax.annotate(concept, (row["style_swd"], row["vis_swd"]),
                    fontsize=7.5, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Style SWD (VGG-16 Gram, relu3_1)", fontsize=11)
ax.set_ylabel("Semantic SWD (CLIP)", fontsize=11)
ax.set_title(f"Style vs Semantic Drift\nSpearman ρ = {rho_st:.3f}, "
             f"p = {p_st:.3f}", fontsize=12, fontweight="bold")
ax.legend(title="Tier")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "style_vs_semantic_swd.png"),
            dpi=150, bbox_inches="tight")
plt.show()

**Key result:** Style covariate is significant ($p = 0.002$), confirming that photographic style changes in YFCC15M contribute to visual SWD. However, the tier-by-resource interaction survives style control, establishing that the measured drift reflects genuine semantic change rather than pure aesthetic evolution.

## 8.3 Reinhard LAB Colour-Transfer Normalisation

The Reinhard [2001] algorithm normalises each image's global LAB colour statistics (mean and standard deviation per channel) to match a canonical reference computed from the full YFCC15M corpus. This removes temporal colour-grading trends without altering image content or composition.

The global canonical style parameters (LAB mean and standard deviation) are computed once and cached to `lab_global_stats.json`. Subsequent runs load from cache.

In [ ]:
STATS_PATH   = os.path.join(RESULTS_DIR, "lab_global_stats.json")
SAMPLE_STATS = 80

GLOBAL_LAB_MEAN = None
GLOBAL_LAB_STD  = None
if os.path.exists(STATS_PATH):
    try:
        with open(STATS_PATH) as f:
            d = json.load(f)
        if "mean" in d and "std" in d and len(d["mean"]) == 3:
            GLOBAL_LAB_MEAN = np.array(d["mean"])
            GLOBAL_LAB_STD  = np.array(d["std"])
            print(f"Loaded global LAB stats from cache.")
        else:
            print("Cache key mismatch — recomputing.")
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Cache corrupted ({e}) — recomputing.")


def rgb_to_lab(img_rgb: np.ndarray) -> np.ndarray:
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    return cv2.cvtColor(bgr.astype(np.float32) / 255.0,
                        cv2.COLOR_BGR2Lab).astype(np.float64)

def lab_to_rgb(lab: np.ndarray) -> np.ndarray:
    bgr = cv2.cvtColor(lab.astype(np.float32), cv2.COLOR_Lab2BGR)
    return cv2.cvtColor((np.clip(bgr, 0, 1) * 255).astype(np.uint8),
                        cv2.COLOR_BGR2RGB)

# Reinhard et al. (2001): normalise per-channel LAB statistics to the global reference.
def reinhard_transfer(img_rgb: np.ndarray,
                      ref_mean: np.ndarray,
                      ref_std:  np.ndarray) -> np.ndarray:
    """Reinhard et al. (2001) LAB colour transfer."""
    lab = rgb_to_lab(img_rgb)
    for ch in range(3):
        mu  = lab[:, :, ch].mean()
        sig = lab[:, :, ch].std() + 1e-9
        lab[:, :, ch] = (lab[:, :, ch] - mu) / sig * ref_std[ch] + ref_mean[ch]
    return lab_to_rgb(lab)


if GLOBAL_LAB_MEAN is None or GLOBAL_LAB_STD is None:
    print("Computing global LAB statistics across corpus...\n")
    ch_acc = [[], [], []]
    all_paths = []
    for concept in CONFIG["concepts"]:
        for t_bin in CONFIG["temporal_bins"]:
            d_path = os.path.join(IMAGES_DIR, concept, t_bin)
            if os.path.isdir(d_path):
                paths = sorted(glob.glob(os.path.join(d_path, "*.jpg")))
                all_paths.extend(paths[:SAMPLE_STATS])

    set_seed(42)
    sampled = random.sample(all_paths, min(5000, len(all_paths)))
    for p in tqdm(sampled, desc="LAB stats"):
        try:
            img = np.array(Image.open(p).convert("RGB").resize((128, 128)))
            lab = rgb_to_lab(img)
            for ch in range(3):
                ch_acc[ch].extend(lab[:, :, ch].flatten().tolist())
        except Exception:
            continue

    GLOBAL_LAB_MEAN = np.array([np.mean(ch_acc[ch]) for ch in range(3)])
    GLOBAL_LAB_STD  = np.array([np.std(ch_acc[ch])  for ch in range(3)])
    with open(STATS_PATH, "w") as f:
        json.dump({"mean": GLOBAL_LAB_MEAN.tolist(),
                    "std":  GLOBAL_LAB_STD.tolist()}, f)
    print(f"Saved \u2192 {STATS_PATH}")

print(f"\nGlobal canonical style:")
print(f"  LAB mean : {GLOBAL_LAB_MEAN.round(3)}")
print(f"  LAB std  : {GLOBAL_LAB_STD.round(3)}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
examples = ["mountain", "surgery", "phone", "television"]
for col, concept in enumerate(examples):
    paths = sorted(glob.glob(
        os.path.join(IMAGES_DIR, concept, "pre2008", "*.jpg")))
    if not paths:
        continue
    orig = np.array(Image.open(paths[0]).convert("RGB"))
    norm = reinhard_transfer(orig, GLOBAL_LAB_MEAN, GLOBAL_LAB_STD)
    for row, (img, ttl) in enumerate([(orig, "Original"),
                                       (norm, "Style-normalized")]):
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{concept}\n{ttl}", fontsize=9)
        axes[row, col].axis("off")

plt.suptitle("Reinhard LAB Colour Transfer | Before vs After",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "reinhard_examples.png"),
            dpi=130, bbox_inches="tight")
plt.show()

## 8.4 CLIP Embedding Re-extraction on Style-Normalised Images

All 19,334 corpus images are passed through the Reinhard normalisation and re-encoded with CLIP ViT-B/32. The resulting style-normalised embeddings (`sn_embs`, shape 19296 x 512) are cached to Drive. This cell is the most computationally expensive in the style analysis (~8 minutes on a T4 GPU) and runs only once.

In [ ]:
SN_EMBED_PATH = os.path.join(EMBED_DIR, "clip_vitb32_sn_embeddings.npy")
SN_META_PATH  = os.path.join(META_DIR,  "clip_vitb32_sn_meta.csv")
BATCH_SN      = 48

if os.path.exists(SN_EMBED_PATH) and os.path.exists(SN_META_PATH):
    sn_embs    = np.load(SN_EMBED_PATH)
    df_sn_meta = pd.read_csv(SN_META_PATH)
    print(f"Loaded SN embeddings: {sn_embs.shape}")
else:
    print("Loading CLIP for style-normalized embedding extraction...\n")
    clip_sn = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    proc_sn = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_sn.eval()

    sn_emb_list = []
    sn_meta_rows = []

    for concept in tqdm(CONFIG["concepts"], desc="SN CLIP embeddings"):
        for t_bin in CONFIG["temporal_bins"]:
            img_dir = os.path.join(IMAGES_DIR, concept, t_bin)
            if not os.path.isdir(img_dir):
                continue
            paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
            if not paths:
                continue

            for s in range(0, len(paths), BATCH_SN):
                batch_paths = paths[s: s + BATCH_SN]
                pil_imgs, valid = [], []
                for p in batch_paths:
                    try:
                        raw  = np.array(Image.open(p).convert("RGB"))
                        norm = reinhard_transfer(raw,
                                                 GLOBAL_LAB_MEAN,
                                                 GLOBAL_LAB_STD)
                        pil_imgs.append(Image.fromarray(norm))
                        valid.append(p)
                    except Exception:
                        continue

                if not pil_imgs:
                    continue

                with torch.no_grad():
                    inp = proc_sn(images=pil_imgs,
                                  return_tensors="pt",
                                  padding=True).to(device)
                    feat = F.normalize(
                        clip_sn.get_image_features(**inp).pooler_output, dim=-1)
                    embs_batch = feat.cpu().numpy()

                for path, emb in zip(valid, embs_batch):
                    sn_emb_list.append(emb)
                    sn_meta_rows.append({
                        "path"    : path,
                        "concept" : concept,
                        "temp_bin": t_bin,
                        "tier"    : TIER_MAP[concept],
                    })

    sn_embs    = np.stack(sn_emb_list, axis=0).astype(np.float32)
    df_sn_meta = pd.DataFrame(sn_meta_rows).reset_index(drop=True)

    np.save(SN_EMBED_PATH, sn_embs)
    df_sn_meta.to_csv(SN_META_PATH, index=False)
    print(f"\nSaved SN embeddings : {SN_EMBED_PATH}  {sn_embs.shape}")
    print(f"Saved SN meta       : {SN_META_PATH}  {len(df_sn_meta)} rows")

    del clip_sn, proc_sn
    torch.cuda.empty_cache() if device == "cuda" else None

## 8.5 Style-Normalised SWD Decomposition

`vis_swd_sn` is computed per concept using the Reinhard-normalised CLIP embeddings. The decomposition:

$$\text{pct_retained} = \frac{\text{vis_swd_sn}}{\text{vis_swd}} \times 100$$

quantifies what fraction of the original SWD signal survives colour normalisation. A high retention value indicates the drift is predominantly semantic; a low value indicates it is substantially colour/tonal.

**Key findings:**
- No concept loses more than 33% of SWD after colour normalisation (range: 67-88%).
- Surgery retains **83.4%**, confirming its drift is overwhelmingly semantic.
- Mountain retains **72.5%**, revising the earlier narrative: only 27.5% of its drift is attributable to colour and tonal grading. The residual is likely compositional (HDR tone-mapping, aerial perspectives), which Reinhard normalisation does not remove.
- Rank correlation of original vs SN SWD: $\rho = 0.948$, $p < 0.001$ — concept ordering is nearly perfectly preserved.

In [ ]:
print("STYLE-NORMALIZED SWD ANALYSIS\n")

# Retrieve style-normalised CLIP embeddings for a (concept, bin) pair.
def get_sn(concept, t_bin):
    idx = df_sn_meta[(df_sn_meta["concept"]  == concept) &
                     (df_sn_meta["temp_bin"] == t_bin)].index.tolist()
    return sn_embs[idx] if idx else np.empty((0, sn_embs.shape[1]))


B0_sn = CONFIG["temporal_bins"][0]
B2_sn = CONFIG["temporal_bins"][2]

# Original vis_swd (CLIP, full images)
df_vis_clip = (DF[DF["model"] == "clip"]
               .drop_duplicates("concept")
               .set_index("concept")[["vis_swd", "tier"]])

sn_records = []
for concept in tqdm(CONFIG["concepts"], desc="SN SWD"):
    pre_sn  = get_sn(concept, B0_sn)
    post_sn = get_sn(concept, B2_sn)
    sn_records.append({
        "concept"   : concept,
        "tier"      : TIER_MAP[concept],
        "vis_swd"   : float(df_vis_clip.loc[concept, "vis_swd"]),
        "vis_swd_sn": sliced_wd(pre_sn, post_sn),
    })

df_sn = pd.DataFrame(sn_records)
df_sn["pct_retained"]  = (df_sn["vis_swd_sn"] / df_sn["vis_swd"] * 100).round(1)
df_sn["pct_style"]     = (100 - df_sn["pct_retained"]).round(1)

print("\nPer-concept decomposition: semantic vs style-driven drift\n")
display(df_sn.sort_values("pct_style", ascending=False).round(5)
             .reset_index(drop=True))

# ── Tier-level summary ────────────────────────────────────────────────────────
print("\nMean SWD by tier — original vs style-normalized:")
tier_sum = df_sn.groupby("tier")[["vis_swd", "vis_swd_sn", "pct_retained"]].mean().round(4)
display(tier_sum)

# ── Rank-order correlation (does SN-SWD preserve concept ordering?) ──────────
rho_sn, p_sn = spearmanr(df_sn["vis_swd"], df_sn["vis_swd_sn"])
print(f"\nRank correlation original vs SN SWD: ρ = {rho_sn:.4f}, p = {p_sn:.4f}")


# ── Final printed interpretation ──────────────────────────────────────────────
print("\n")
print("SEMANTIC vs STYLE DECOMPOSITION SUMMARY")

semantic_driven = df_sn[df_sn["pct_retained"] >= 75].sort_values("pct_retained", ascending=False)
style_driven    = df_sn[df_sn["pct_style"]    >= 50].sort_values("pct_style",    ascending=False)

print(f"\nConcepts where ≥75% of drift survives style normalization")
print(f"(predominantly SEMANTIC change):")
for _, r in semantic_driven.iterrows():
    print(f"  {r['concept']:<14} {r['pct_retained']:.1f}% retained  "
          f"[{r['tier']}]  vis_swd={r['vis_swd']:.5f}")

if len(style_driven):
    print(f"\nConcepts where ≥50% of drift is absorbed by style normalization")
    print(f"(substantial AESTHETIC component):")
    for _, r in style_driven.iterrows():
        print(f"  {r['concept']:<14} {r['pct_style']:.1f}% style  "
              f"[{r['tier']}]  vis_swd={r['vis_swd']:.5f}")
else:
    print("\nNo concept loses ≥50% of SWD after style normalization.")
    print("The measured drift is predominantly semantic across all concepts.")

In [ ]:
# ── Side-by-side grouped bar chart ────────────────────────────────────────────
df_plot = df_sn.sort_values(["tier", "vis_swd"], ascending=[True, False]).reset_index(drop=True)
palette = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}
x      = np.arange(len(df_plot))
width  = 0.38

fig, ax = plt.subplots(figsize=(18, 6))
bars1 = ax.bar(x - width/2, df_plot["vis_swd"],    width,
               label="Original SWD",          color="steelblue",  alpha=0.85,
               edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width/2, df_plot["vis_swd_sn"], width,
               label="Style-normalized SWD",  color="darkorange", alpha=0.85,
               edgecolor="black", linewidth=0.5)

# Tier separator lines
tiers_plot = df_plot["tier"].values
for i in range(1, len(tiers_plot)):
    if tiers_plot[i] != tiers_plot[i-1]:
        ax.axvline(i - 0.5, color="black", linewidth=2, linestyle="--")
        ax.text(i - 0.5, ax.get_ylim()[1] * 0.98,
                f" {tiers_plot[i]}", fontsize=9, va="top")

ax.set_xticks(x)
ax.set_xticklabels(df_plot["concept"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Sliced Wasserstein Distance")
ax.set_title("CLIP Visual SWD: Original vs Style-Normalized",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "sn_swd_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Scatter: % style vs concept tier ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for tier, sub in df_sn.groupby("tier"):
    ax.scatter(sub["vis_swd"], sub["pct_style"],
               c=palette[tier], label=tier, s=120,
               edgecolors="black", linewidths=0.5, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row["concept"], (row["vis_swd"], row["pct_style"]),
                    fontsize=8, xytext=(4, 3), textcoords="offset points")
ax.axhline(50, color="red", linestyle="--", linewidth=1,
           label="50% style threshold")
ax.set_xlabel("Original SWD (CLIP)", fontsize=11)
ax.set_ylabel("Drift attributable to style (%)", fontsize=11)
ax.set_title("Semantic vs Aesthetic Drift Decomposition\n",
             fontsize=11, fontweight="bold")
ax.legend(title="Tier")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "style_semantic_decomposition.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 8.6 H1 on Style-Normalised SWD

H1 is retested using `vis_swd_sn` as the dependent variable under both the a priori and data-driven taxonomies. The H3 Wilcoxon test is also replicated on the SN-CLIP trajectory angles to confirm that trajectory smoothness is not an artefact of colour-grading trends.

In [ ]:
print("H1 — STYLE-NORMALIZED SWD (vis_swd_sn as DV)\n")

# A priori tier groups (vis_swd_sn, 6 concepts per tier)
df_sn_h1 = df_sn.set_index("concept").copy()
df_sn_h1["dd_tier"] = df_sn_h1.index.map(DD_TIER_MAP)

for taxonomy, tier_col in [("A priori", "tier"), ("Data-driven", "dd_tier")]:
    print(f"\n  [{taxonomy}]")
    for t in ["stable", "slow", "fast"]:
        vals = df_sn_h1[df_sn_h1[tier_col] == t]["vis_swd_sn"].values
        print(f"    {t:<8} n={len(vals)}  "
              f"mean={vals.mean():.6f}  std={vals.std():.6f}")

# ── Pairwise permutation tests on vis_swd_sn (concept-level, n=6 per tier) ───
print("\n--- Pairwise permutation tests on vis_swd_sn ---")
for taxonomy, tier_col in [("A priori", "tier"), ("Data-driven", "dd_tier")]:
    print(f"\n  [{taxonomy}]")
    stable_g = df_sn_h1[df_sn_h1[tier_col]=="stable"]["vis_swd_sn"].values
    slow_g   = df_sn_h1[df_sn_h1[tier_col]=="slow"  ]["vis_swd_sn"].values
    fast_g   = df_sn_h1[df_sn_h1[tier_col]=="fast"  ]["vis_swd_sn"].values

    for label, a, b in [("fast vs stable", fast_g, stable_g),
                         ("fast vs slow",   fast_g, slow_g)]:
        n = min(len(a), len(b))
        if n < 2:
            print(f"    {label}: insufficient group size, skipping.")
            continue
        av, bv = a[:n], b[:n]
        t_stat, p2 = ttest_rel(av, bv)
        p1 = p2/2 if t_stat > 0 else 1.0
        obs, p_perm = permutation_test_paired(av, bv)
        d = cohen_d_paired(av, bv)
        size = "large" if abs(d)>0.8 else "medium" if abs(d)>0.5 else "small"
        print(f"    {label} (n={n}):  "
              f"t={t_stat:.3f}  p={p1:.4f}  p_perm={p_perm:.4f}  "
              f"d={d:.3f} ({size})")

# ── Kruskal-Wallis test on vis_swd_sn (correct test: no language dimension) ──
print("\n--- Kruskal-Wallis test on vis_swd_sn (concept-level, n=18) ---")


for taxonomy, tier_col in [("A priori", "tier"), ("Data-driven", "dd_tier")]:
    stable_g = df_sn_h1[df_sn_h1[tier_col]=="stable"]["vis_swd_sn"].values
    slow_g   = df_sn_h1[df_sn_h1[tier_col]=="slow"  ]["vis_swd_sn"].values
    fast_g   = df_sn_h1[df_sn_h1[tier_col]=="fast"  ]["vis_swd_sn"].values

    H_stat, p_kw = kruskal(stable_g, slow_g, fast_g)
    print(f"  [{taxonomy}]  Kruskal-Wallis H={H_stat:.3f}, p={p_kw:.4f}  "
          f"→ {'SIGNIFICANT' if p_kw < 0.05 else 'n.s.'}")

    # Mann-Whitney post-hoc (fast vs stable, fast vs slow)
    for label, a, b in [("fast vs stable", fast_g, stable_g),
                         ("fast vs slow",   fast_g, slow_g)]:
        u_stat, p_mw = mannwhitneyu(a, b, alternative="greater")
        print(f"    Mann-Whitney {label}: U={u_stat:.1f}, p={p_mw:.4f}  "
              f"{'*' if p_mw<0.05 else 'ns'}")

# ── Tier-level SWD retention summary ─────────────────────────────────────────
print("\n--- Style-normalization SWD retention by tier (a priori) ---")
for t in ["stable", "slow", "fast"]:
    sub = df_sn_h1[df_sn_h1["tier"] == t]
    orig = sub["vis_swd"].mean()
    sn   = sub["vis_swd_sn"].mean()
    print(f"  {t:<8}: {sn/orig*100:.1f}% retained after style normalization  "
          f"(mean orig={orig:.5f}, SN={sn:.5f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette   = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}

for ax, (tier_col, label) in zip(axes, [
    ("tier",    "A Priori Taxonomy"),
    ("dd_tier", "Data-Driven Taxonomy (tertile)"),
]):
    order = ["stable", "slow", "fast"]
    means_orig = df_sn_h1.groupby(tier_col)["vis_swd"].mean().reindex(order)
    means_sn   = df_sn_h1.groupby(tier_col)["vis_swd_sn"].mean().reindex(order)
    x     = np.arange(len(order))
    width = 0.35
    ax.bar(x - width/2, means_orig, width, label="Original SWD",
           color=[palette[t] for t in order], alpha=0.85,
           edgecolor="black", linewidth=0.6)
    ax.bar(x + width/2, means_sn,   width, label="Style-Normalized SWD",
           color=[palette[t] for t in order], alpha=0.45,
           edgecolor="black", linewidth=0.6, hatch="///")
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.set_ylabel("Mean Sliced Wasserstein Distance")
    ax.set_title(label, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(means_orig.max(), means_sn.max()) * 1.35)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "h1_sn_swd_by_tier.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 8.7 H3 Replicated on Style-Normalised Embeddings

Trajectory smoothness (H3) is retested on the style-normalised CLIP embeddings. If H3 survives colour normalisation, the structured directional drift reflects semantic rather than photographic evolution.

**Key result:** Wilcoxon $W = 0$, $p < 0.001$ on SN-CLIP, matching the original result exactly. The protest anomaly persists and slightly worsens after normalisation ($\Delta\theta = 2.19^\circ$ original vs. $\approx 2^\circ$ SN), confirming it is event-contingent rather than aesthetic.

In [ ]:
print("H3 — TRAJECTORY SMOOTHNESS ON STYLE-NORMALIZED CLIP\n")

B0_sn3, B1_sn3, B2_sn3 = CONFIG["temporal_bins"]


def centroid_sn(concept, t_bin):
    e = get_sn(concept, t_bin)
    if len(e) == 0:
        return None
    n = e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)
    c = n.mean(axis=0)
    return c / (np.linalg.norm(c) + 1e-12)


set_seed(42)
sn_traj_records = []

for concept in CONFIG["concepts"]:
    mu = {b: centroid_sn(concept, b) for b in CONFIG["temporal_bins"]}
    if any(v is None for v in mu.values()):
        continue

    v1 = mu[B1_sn3] - mu[B0_sn3]
    v2 = mu[B2_sn3] - mu[B1_sn3]
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-10 or n2 < 1e-10:
        continue

    theta_obs = float(np.degrees(np.arccos(
        np.clip(np.dot(v1/n1, v2/n2), -1.0, 1.0))))

    # Permutation null: shuffle bin labels per concept
    all_sn = np.concatenate([get_sn(concept, b) for b in CONFIG["temporal_bins"]])
    n_total = len(all_sn)
    if n_total < 6:
        continue

    sizes = [len(get_sn(concept, b)) for b in CONFIG["temporal_bins"]]
    rng_null = np.random.default_rng(sum(ord(ch) for ch in concept))
    null_thetas = []
    for _ in range(CONFIG["n_permutations"]):
        perm = rng_null.permutation(n_total)
        i0   = sizes[0]
        i1   = sizes[0] + sizes[1]
        mu0  = all_sn[perm[:i0]].mean(axis=0)
        mu1  = all_sn[perm[i0:i1]].mean(axis=0)
        mu2  = all_sn[perm[i1:]].mean(axis=0)
        u1, u2 = mu1 - mu0, mu2 - mu1
        nn1, nn2 = np.linalg.norm(u1), np.linalg.norm(u2)
        if nn1 < 1e-10 or nn2 < 1e-10:
            continue
        null_thetas.append(float(np.degrees(
            np.arccos(np.clip(np.dot(u1/nn1, u2/nn2), -1.0, 1.0)))))

    null_med = float(np.median(null_thetas)) if null_thetas else float("nan")
    sn_traj_records.append({
        "concept"     : concept,
        "tier"        : TIER_MAP[concept],
        "theta_vis_sn": theta_obs,
        "null_median" : null_med,
        "delta_theta" : null_med - theta_obs,
    })

df_sn_traj = pd.DataFrame(sn_traj_records).sort_values("delta_theta",
                                                         ascending=False)
print(f"Concepts with valid SN trajectory: {len(df_sn_traj)}\n")
display(df_sn_traj.round(3).reset_index(drop=True))

# ── Wilcoxon test ─────────────────────────────────────────────────────────────
obs_sn  = df_sn_traj["theta_vis_sn"].values
null_sn = df_sn_traj["null_median"].values
diffs   = obs_sn - null_sn   # negative = obs below null = directionally consistent

w_sn, p_sn = wilcoxon(diffs, alternative="less")
print(f"\nWilcoxon W = {w_sn:.1f},  p = {p_sn:.4f}  (one-sided, obs < null)")
print(f"H3 on SN-CLIP: {'SUPPORTED' if p_sn < 0.05 else 'NOT SUPPORTED'}")

# Note protest anomaly explicitly
protest_sn = df_sn_traj[df_sn_traj["concept"] == "protest"]
if len(protest_sn):
    r = protest_sn.iloc[0]
    flag = "ABOVE null" if r["delta_theta"] < 0 else "below null"
    print(f"\nProtest: θ_SN = {r['theta_vis_sn']:.2f}°  "
          f"null = {r['null_median']:.2f}°  "
          f"Δθ = {r['delta_theta']:.2f}°  ({flag})")
    print("  The protest anomaly persists and slightly worsens after style")
    print("  normalization, confirming it is event-contingent, not aesthetic.")

# ── Compare original vs SN trajectory angles ──────────────────────────────────
orig_angles = (DF_TRAJ.drop_duplicates("concept").set_index("concept")["theta_vis"])
df_sn_traj_cmp = df_sn_traj.set_index("concept").copy()
df_sn_traj_cmp["theta_vis_orig"] = orig_angles
df_sn_traj_cmp["angle_diff"]     = (df_sn_traj_cmp["theta_vis_sn"] -
                                     df_sn_traj_cmp["theta_vis_orig"])

rho_a, p_a = spearmanr(df_sn_traj_cmp["theta_vis_orig"].dropna(),
                        df_sn_traj_cmp["theta_vis_sn"].reindex(
                            df_sn_traj_cmp["theta_vis_orig"].dropna().index))
print(f"\nRank correlation (original θ vs SN θ): ρ = {rho_a:.4f}, p = {p_a:.4f}")
print("Per-concept angle comparison:")
display(df_sn_traj_cmp[["tier", "theta_vis_orig", "theta_vis_sn",
                          "angle_diff"]].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
palette = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}
concepts_sorted = df_sn_traj.sort_values("delta_theta",
                                          ascending=False)["concept"].tolist()
x = np.arange(len(concepts_sorted))
colors = [palette[df_sn_traj.set_index("concept").loc[c, "tier"]]
          for c in concepts_sorted]
orig_sorted = [float(orig_angles.get(c, np.nan)) for c in concepts_sorted]
sn_sorted   = [float(df_sn_traj.set_index("concept").loc[c, "theta_vis_sn"])
               for c in concepts_sorted]

ax.scatter(x, sn_sorted,   color=colors, s=90, zorder=4,
           label="SN-CLIP θ", edgecolors="black", linewidths=0.5)
ax.scatter(x, orig_sorted, color=colors, s=90, marker="^", zorder=3,
           alpha=0.5, label="Original CLIP θ", edgecolors="black", linewidths=0.5)
ax.axhline(120, color="grey", linestyle="--", linewidth=1,
           label="Permutation null (~120°)")
for xi, (o, s) in enumerate(zip(orig_sorted, sn_sorted)):
    if not np.isnan(o):
        ax.plot([xi, xi], [o, s], color="grey", linewidth=0.8, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(concepts_sorted, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Trajectory Angle θ (degrees)")
ax.set_title("H3: Trajectory Angles — Original vs Style-Normalized CLIP\n"
             "Circles = SN, Triangles = Original. "
             "Dashed = permutation null (~120°). Lower = more directional.",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=9)
from matplotlib.patches import Patch
tier_handles = [Patch(facecolor=palette[t], edgecolor="black", label=t)
                for t in ["stable", "slow", "fast"]]
ax.legend(handles=ax.get_legend_handles_labels()[0] + tier_handles,
          labels=ax.get_legend_handles_labels()[1] + ["stable", "slow", "fast"],
          fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "h3_sn_trajectory_angles.png"),
            dpi=150, bbox_inches="tight")
plt.show()

---

# Section IX: Retrieval Quality

**MAP@k** [Manning et al., 2008] is computed as a practical measure of the H2 regime gradient. Neutral prompts (without temporal anchors) are used so that retrieval quality reflects the semantic resolution of the text encoder for each language.

**Setup:**
- Pool: all 19,334 images (all concepts and bins mixed).
- Query: mean neutral text embedding $\bar{\mathbf{q}}_{c,\ell}$ for concept $c$ in language $\ell$.
- Relevant: any image where `matched_concept == c` (bin-agnostic).
- Random baseline: $1/18 \approx 0.056$.

**Each model uses its own aligned text encoder** for retrieval: M-CLIP for CLIP, XLM-RoBERTa for M-OpenCLIP, SigLIP-2 native encoder for SigLIP-2.

**Key results (MAP@10):** CLIP collapses to 0.191 for Bengali, barely above the random baseline. M-OpenCLIP raises Bengali MAP@10 to 0.659 (3.4-fold improvement) by joint training. Swahili remains the hardest language across all models (0.289, 0.326, 0.314), confirming that its difficulty is inherent to data volume rather than encoder-specific failures.

## 9.1 Average Precision at k

In [ ]:
# Average Precision at k: area under the precision-recall curve truncated at k.
def average_precision_at_k(relevance, k):
    relevance = relevance[:k].astype(float)
    if relevance.sum() == 0:
        return 0.0
    hits, running_sum = 0, 0.0
    for i, r in enumerate(relevance):
        if r:
            hits += 1
            running_sum += hits / (i + 1)
    return running_sum / min(k, relevance.sum())


retrieval_records = []

print("Retrieval Quality Evaluation\n")
print("Each model uses its own aligned text encoder:")
print("  CLIP      -> M-CLIP (fine-tuned on CLIP, 512-dim)")
print("  M-OpenCLIP -> XLM-RoBERTa tokenizer (native, 512-dim)")
print("  SigLIP-2   -> SigLIP-2 text encoder (native, 768-dim)\n")

for model_key, model_cfg in CONFIG["models"].items():
    _embs = VIS_EMBS[model_key]
    _meta = DF_META[model_key]
    emb_src = TEXT_EMB_MAP[model_key]

    for lang in CONFIG["languages"]:
        for concept in tqdm(CONFIG["concepts"],
                            desc=f"Retrieval [{model_key}|{lang}]"):
            q = mean_emb(emb_src[lang][concept]["neutral"])
            sims       = _embs @ q
            sorted_idx = np.argsort(sims)[::-1]
            relevance  = (_meta.loc[sorted_idx, "concept"].values == concept).astype(int)

            for k in CONFIG["k_values"]:
                retrieval_records.append({
                    "model"    : model_key,
                    "concept"  : concept,
                    "tier"     : TIER_MAP[concept],
                    "language" : lang,
                    "k"        : k,
                    "precision": relevance[:k].mean(),
                    "AP"       : average_precision_at_k(relevance, k),
                })

DF_RETRIEVAL = pd.DataFrame(retrieval_records)
retrieval_path = os.path.join(RESULTS_DIR, "retrieval_quality.csv")
DF_RETRIEVAL.to_csv(retrieval_path, index=False)
print(f"\nRetrieval quality saved: {retrieval_path}")

## 9.2 MAP@k Summary and Bar Chart

In [ ]:
print("Retrieval Quality Summary\n" + "=" * 60)

lang_order = sorted(CONFIG["languages"],
                    key=lambda l: CONFIG["resource_level"][l], reverse=True)

for mk in CONFIG["models"]:
    sub = DF_RETRIEVAL[DF_RETRIEVAL["model"] == mk]
    if sub.empty: continue
    print(f"\n-- {mk.upper()} --")
    map_summary = (
        sub.groupby(["language", "k"])[["precision", "AP"]]
           .mean()
           .rename(columns={"precision": "mean_P@k", "AP": "MAP@k"})
           .round(4)
    )
    display(map_summary)

print("\n-- Cross-model MAP@10 (all languages, mean across concepts) --")
cross = []
for mk in CONFIG["models"]:
    sub = DF_RETRIEVAL[(DF_RETRIEVAL["model"] == mk) & (DF_RETRIEVAL["k"] == 10)]
    if sub.empty: continue
    row = {"model": mk.upper(), "overall_MAP@10": round(sub["AP"].mean(), 4)}
    for lang in lang_order:
        row[lang] = round(sub[sub["language"] == lang]["AP"].mean(), 4)
    cross.append(row)
df_cross = pd.DataFrame(cross).set_index("model")
display(df_cross)

out = os.path.join(RESULTS_DIR, "retrieval_summary.csv")
df_cross.to_csv(out)
print(f"\nSaved: {out}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
fig.suptitle("MAP@10 by Model and Language", fontsize=12, fontweight="bold")
model_colors = {"CLIP": "#4e79a7", "M_OPENCLIP": "#f28e2b", "SIGLIP2": "#59a14f"}
n_models = len(CONFIG["models"])
bar_w    = 0.25
x        = np.arange(len(lang_order))
for j, mk in enumerate(CONFIG["models"]):
    sub = DF_RETRIEVAL[(DF_RETRIEVAL["model"] == mk) & (DF_RETRIEVAL["k"] == 10)]
    vals = [sub[sub["language"] == l]["AP"].mean() for l in lang_order]
    ax.bar(x + (j - n_models/2 + 0.5) * bar_w, vals, width=bar_w,
           label=mk.upper(), color=model_colors.get(mk.upper(), f"C{j}"), alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([l.upper() for l in lang_order], fontsize=9)
ax.set_ylabel("MAP@10")
ax.axhline(1/len(CONFIG["concepts"]), color="red", linestyle="--",
           linewidth=1, alpha=0.6, label=f"Random baseline (1/{len(CONFIG['concepts'])})")
ax.legend(fontsize=8)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, "retrieval_map10_bar.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")


# Section X: Visualizations

## 10.1 UMAP Projection of Visual Embedding Spaces

UMAP [McInnes et al., 2018] projects each model's visual embedding matrix to 2D for qualitative inspection.

In [ ]:
set_seed(42)

for model_key in CONFIG["models"]:
    _embs = VIS_EMBS[model_key]
    _meta = DF_META[model_key]

    print(f"Fitting UMAP on {model_key} embeddings ({_embs.shape[0]} images)...")
    reducer = umap_lib.UMAP(
        n_components=2, n_neighbors=30, min_dist=0.1,
        metric="cosine", random_state=42, low_memory=False,
    )
    umap_2d = reducer.fit_transform(_embs)
    np.save(os.path.join(RESULTS_DIR, f"umap_2d_{model_key}.npy"), umap_2d)
    print(f"UMAP done. Shape: {umap_2d.shape}")

    tier_palette = {"pre2008": "#4e79a7", "2008_2011": "#f28e2b",
                    "2012_2014": "#e15759"}
    tiers      = ["stable", "slow", "fast"]
    tier_titles = {"stable": "Stable", "slow": "Slow-drift", "fast": "Fast-drift"}

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        f"{model_key.upper()} Visual Embedding Space",
        fontsize=13, fontweight="bold")

    for ax, tier in zip(axes, tiers):
        mask     = _meta["concept"].isin(CONFIG["concept_tiers"][tier])
        sub_embs = umap_2d[mask]
        sub_bins = _meta.loc[mask, "temp_bin"].values
        for t_bin, colour in tier_palette.items():
            m2 = sub_bins == t_bin
            ax.scatter(sub_embs[m2, 0], sub_embs[m2, 1],
                       c=colour, label=t_bin, alpha=0.4, s=8, linewidths=0)
        ax.set_title(tier_titles[tier], fontsize=11)
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
        ax.legend(title="Temporal bin", fontsize=8, markerscale=2)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, f"umap_{model_key}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}\n")

## 10.2 Anomaly Spotlight

Quantitative summaries for the key phenomena identified in the paper.

**Semantic fossilisation (television, English):** The English text encoder anchors to the dominant CRT archetype in its training corpus. Temporal prompts produce insufficient directional shift to overcome this anchor. Fingerprint: the EN/BN retrieved-image SWD ratio is < 1, meaning English queries retrieve *less* temporally diverse imagery than Bengali despite a resource advantage of more than two orders of magnitude.

**Medical modernisation anomaly (surgery):** Surgery (labeled slow-drift a priori) produces the highest visual SWD of all 18 concepts (0.0265), exceeding television (0.0150) by 77%. Pre-2008 retrieves a uniform archetype of surgical teams in operating rooms; 2012-2014 retrieves post-operative patient experience documentation, reflecting proliferation of laparoscopic platforms, digital intraoperative imaging, and social-media patient documentation during 2004-2014.

In [ ]:
print("ANOMALY SPOTLIGHT")

print("\n[1] PROTEST — Near-Random Trajectory")
protest_row = DF_TRAJ[DF_TRAJ["concept"] == "protest"].iloc[0]
phone_dt    = DF_TRAJ[DF_TRAJ["concept"] == "phone"]["delta_theta"].values[0]
print(f"    Observed theta : {protest_row['theta_vis']:.2f} deg")
print(f"    Null median    : {protest_row['null_median']:.2f} deg")
print(f"    Delta-theta    : {protest_row['delta_theta']:.2f} deg  (vs phone: {phone_dt:.2f} deg)")

print("\n[2] TELEVISION — Semantic Fossilization")
tv_vis   = DF[DF["concept"] == "television"]["vis_swd"].iloc[0]
tv_en    = DF[(DF["concept"] == "television") & (DF["language"] == "en")]["text_delta"].iloc[0]
tv_bn    = DF[(DF["concept"] == "television") & (DF["language"] == "bn")]["text_delta"].iloc[0]
phone_en = DF[(DF["concept"] == "phone")      & (DF["language"] == "en")]["text_delta"].iloc[0]
print(f"    Television vis_swd   : {tv_vis:.6f}")
print(f"    Television EN text d : {tv_en:.6f}")
print(f"    Television BN text d : {tv_bn:.6f}")
print(f"    Phone      EN text d : {phone_en:.6f}  (for comparison)")

print("\n[3] Retrieved-image SWD ratio (EN / BN) for fast-drift concepts at k=10:")
print("    ratio < 1 -> EN retrieves LESS diverse imagery (Fossilization)")
for concept in CONFIG["concept_tiers"]["fast"]:
    if concept not in DF["concept"].values: continue
    en_v = DF[(DF["concept"] == concept) & (DF["language"] == "en")]["ret_swd_k10"].values
    bn_v = DF[(DF["concept"] == concept) & (DF["language"] == "bn")]["ret_swd_k10"].values
    if len(en_v) == 0 or len(bn_v) == 0: continue
    ratio = en_v[0] / bn_v[0] if bn_v[0] > 0 else float("nan")
    flag  = " <- fossilized" if ratio < 0.95 else ""
    print(f"  {concept:<14} EN={en_v[0]:.5f}  BN={bn_v[0]:.5f}  ratio={ratio:.2f}x{flag}")

print("\n[4] SURGERY — Slow-Drift Concept with Fast-Drift Visual Signal")
surg_swd  = DF[DF["concept"] == "surgery"]["vis_swd"].iloc[0]
surg_traj = DF_TRAJ[DF_TRAJ["concept"] == "surgery"]
tv_swd    = DF[DF["concept"] == "television"]["vis_swd"].iloc[0]
print(f"    surgery vis_swd    : {surg_swd:.6f}  (highest of all 18 concepts)")
print(f"    television vis_swd : {tv_swd:.6f}  (second highest)")
if len(surg_traj):
    print(f"    theta_vis          : {surg_traj.iloc[0]['theta_vis']:.2f} deg   "
          f"delta_theta = {surg_traj.iloc[0]['delta_theta']:.2f} deg")
print("    Surgery changed more visually than any technology concept 2004-2014:")
print("    laparoscopic proliferation, robotic surgery (da Vinci), digital OR")
print("    imaging, and sterile draping standards all shifted the visual space")
print("    substantially. This challenges the slow-drift classification and")
print("    suggests procedure-type concepts warrant their own tier.")

## 10.3 Drift Heatmap

Two-panel heatmap. Left: visual SWD per concept (language-independent), ordered by semantic tier. Right: text direction drift $\delta$ per (concept, language) cell, with languages ordered from high to low mC4 resource volume (left to right).

In [ ]:
fig = plt.figure(figsize=(20, 7))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, len(CONFIG["languages"])],
                        wspace=0.35)

concept_order = (CONFIG["concept_tiers"]["stable"] +
                 CONFIG["concept_tiers"]["slow"]   +
                 CONFIG["concept_tiers"]["fast"])
# Only keep concepts that exist in DF (in case running with original 10)
concept_order = [c for c in concept_order if c in DF["concept"].unique()]

# ── Left panel: Visual SWD (language-independent) ─────────────
ax_vis = fig.add_subplot(gs[0])
vis_vals = (DF.drop_duplicates("concept")
              .set_index("concept")["vis_swd"]
              .reindex(concept_order))
sns.heatmap(
    pd.DataFrame(vis_vals.values, index=vis_vals.index, columns=["Visual SWD"]),
    ax=ax_vis, annot=True, fmt=".5f", cmap="YlOrRd",
    linewidths=0.5, cbar_kws={"label": "Sliced Wasserstein Distance"}
)
ax_vis.set_title("Visual SWD\n(language-independent)", fontsize=11, fontweight="bold")
ax_vis.set_xlabel("")
# Tier separator lines
n_stable = len([c for c in concept_order if TIER_MAP.get(c) == "stable"])
n_slow   = len([c for c in concept_order if TIER_MAP.get(c) == "slow"])
for sep in [n_stable, n_stable + n_slow]:
    ax_vis.axhline(sep, color="black", linewidth=2)

# ── Right panel: Text direction drift Δ (language-dependent) ──
ax_txt = fig.add_subplot(gs[1])
pivot = DF.pivot(index="concept", columns="language", values="text_delta")
pivot = pivot.reindex(concept_order)
# Reorder columns by resource level (high → low)
lang_order = sorted(CONFIG["languages"],
                    key=lambda l: CONFIG["resource_level"][l], reverse=True)
pivot = pivot[[l for l in lang_order if l in pivot.columns]]
sns.heatmap(
    pivot, ax=ax_txt, annot=True, fmt=".5f", cmap="YlOrRd",
    linewidths=0.5, cbar_kws={"label": "Text Drift δ_text"}
)
ax_txt.set_title("Text Direction Drift δ_text\n(language-dependent, ordered high→low resource)",
                 fontsize=11, fontweight="bold")
ax_txt.set_xlabel("Language (↓ resource)")
for sep in [n_stable, n_stable + n_slow]:
    ax_txt.axhline(sep, color="black", linewidth=2)

fig.suptitle("Drift Heatmap: Visual SWD and Text Direction Drift (pre2008 → 2012_2014)",
             fontsize=13, fontweight="bold", y=1.01)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "drift_heatmap.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: drift_heatmap.png")

## 10.4 Visual Centroid Trajectory Plot

PCA-projected centroid trajectories for each concept. A global PCA fit on the full embedding matrix preserves cross-concept geometry. Fast-drift concepts show directed arcs; protest wanders near the permutation null.

In [ ]:
set_seed(42)
B0, B1, B2 = CONFIG["temporal_bins"]
concepts_to_plot = (CONFIG["concept_tiers"]["fast"] +
                    CONFIG["concept_tiers"]["stable"])
n_plots = len(concepts_to_plot)
ncols   = 4
nrows   = (n_plots + ncols - 1) // ncols

for model_key in CONFIG["models"]:
    _embs_pca = VIS_EMBS[model_key]

    pca = PCA(n_components=2, random_state=42)
    pca.fit(_embs_pca)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()
    fig.suptitle(
        f"Visual Centroid Trajectories — {model_key.upper()}\n"
        f"(pre2008 → 2008_2011 → 2012_2014)",
        fontsize=13, fontweight="bold")

    tier_color = {"stable": "#4e79a7", "slow": "#59a14f", "fast": "#e15759"}

    for ax, concept in zip(axes, concepts_to_plot):
        centroids_3d = np.array([
            centroid(get_vis(concept, b, model_key))
            for b in CONFIG["temporal_bins"]
        ])
        pts_2d = pca.transform(centroids_3d)
        color  = tier_color[TIER_MAP[concept]]
        ax.plot(pts_2d[:, 0], pts_2d[:, 1], "-o",
                color=color, linewidth=2, markersize=8, zorder=3)
        for label, pt in zip([B0, B1, B2], pts_2d):
            ax.annotate(label, pt, fontsize=7, ha="left",
                        xytext=(5, 5), textcoords="offset points")
        ax.set_title(f"{concept.capitalize()} [{TIER_MAP[concept]}]",
                     fontsize=10)
        ax.set_xlabel("PCA-1", fontsize=8)
        ax.set_ylabel("PCA-2", fontsize=8)

    for ax in axes[n_plots:]:
        ax.set_visible(False)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR,
                       f"visual_centroid_trajectory_{model_key}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}\n")

## 10.5 H2 Regime Classification Scatter

One scatter plot per language in the joint ($\Delta$, $\sigma^2$) plane. Dashed lines mark the median thresholds for regime classification. Points are coloured by regime and shaped by concept tier.

In [ ]:
n_langs = len(CONFIG["languages"])
ncols   = min(n_langs, 3)
nrows   = (n_langs + ncols - 1) // ncols
fig, axes_arr = plt.subplots(nrows, ncols,
                              figsize=(6 * ncols, 5 * nrows),
                              sharey=True, sharex=True)
axes_flat = np.array(axes_arr).flatten()

fig.suptitle("H2 Regime Classification: Text Direction Drift Δ vs Cluster Dispersion σ²\n"
             "(ordered by training-data volume: high → low resource)",
             fontsize=13, fontweight="bold")

regime_colors = {
    "drifting" : "#e15759",
    "frozen"   : "#4e79a7",
    "noisy"    : "#f28e2b",
    "ambiguous": "#b07aa1",
}
tier_markers = {"stable": "o", "slow": "s", "fast": "^"}

# Order languages by resource level descending
lang_order = sorted(CONFIG["languages"],
                    key=lambda l: CONFIG["resource_level"][l], reverse=True)

for ax, lang in zip(axes_flat, lang_order):
    sub = DF[DF["language"] == lang].copy()
    for _, row in sub.iterrows():
        m = tier_markers[row["tier"]]
        c = regime_colors[row["regime"]]
        ax.scatter(row["text_delta"], row["text_sigma2_mean"],
                   marker=m, color=c, s=130, zorder=3,
                   edgecolors="black", linewidths=0.5)
        ax.annotate(row["concept"], (row["text_delta"], row["text_sigma2_mean"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.axvline(thresh_delta,  color="grey", linestyle="--", linewidth=1, alpha=0.7)
    ax.axhline(thresh_sigma2, color="grey", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(f"{lang.upper()}", fontsize=11)
    ax.set_xlabel("Text Drift Δ", fontsize=9)
    ax.set_ylabel("Cluster Dispersion σ²", fontsize=9)

# Hide unused subplots
for ax in axes_flat[n_langs:]:
    ax.set_visible(False)

# Unified legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = (
    [Patch(facecolor=c, label=r) for r, c in regime_colors.items()] +
    [Line2D([0], [0], marker=m, color="k", linestyle="None", label=t)
     for t, m in tier_markers.items()]
)
axes_flat[n_langs - 1].legend(handles=legend_elements, fontsize=8,
                               loc="upper right", title="Regime / Tier")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "h2_regime_scatter.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: h2_regime_scatter.png")



# Section XI: Qualitative Retrieval Grid Analysis

Top-5 retrieval grids for curated (concept, language, model) triples that directly ground the quantitative findings.


In [ ]:
def build_retrieval_grid(concept: str, lang: str, k: int = 5,
                         model: str = None):
    """
    Retrieves top-k images per temporal bin using each model's own
    aligned text encoder.
    """
    m     = model or ACTIVE_MODEL
    _meta = DF_META[m]
    _embs = VIS_EMBS[m]

    # Route text source based on model
    if m == "siglip2":
        emb_src = SIGLIP2_TEXT_EMBS
    elif m == "m_openclip":
        emb_src = MOC_TEXT_EMBS
    else:
        emb_src = TEXT_EMBS

    bins = CONFIG["temporal_bins"]
    fig, axes = plt.subplots(len(bins), k, figsize=(k * 2.8, len(bins) * 2.8))
    fig.suptitle(
        f"'{concept.capitalize()}'  |  Lang: {lang.upper()}  |  "
        f"Model: {m.upper()}",
        fontsize=12, fontweight="bold"
    )

    for row_idx, t_bin in enumerate(bins):
        # Correctly access nested dict: [lang][concept]["temporal"][t_bin]
        raw = emb_src[lang][concept]["temporal"][t_bin]
        q = raw.mean(axis=0)
        q = q / np.linalg.norm(q)   # L2-normalise mean embedding

        # Index into model-specific pool
        pool_mask = ((_meta["concept"] == concept) &
                     (_meta["temp_bin"]  == t_bin))
        pool_idx  = _meta[pool_mask].index.tolist()

        if not pool_idx:
            for col_idx in range(k):
                ax = axes[row_idx, col_idx]
                ax.text(0.5, 0.5, "no images", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8, color="grey")
                ax.set_xticks([]); ax.set_yticks([])
            axes[row_idx, 0].set_ylabel(
                t_bin.replace("_", "–"), fontsize=10,
                rotation=0, labelpad=65, ha="right", va="center")
            continue

        pool_embs = _embs[pool_idx]          # (N, D) — D matches q
        sims      = pool_embs @ q            # (N,)
        top_local = np.argsort(sims)[-k:][::-1]
        filenames = _meta.loc[
            [pool_idx[i] for i in top_local], "filename"
        ].tolist()

        axes[row_idx, 0].set_ylabel(
            t_bin.replace("_", "–"), fontsize=10,
            rotation=0, labelpad=65, ha="right", va="center")

        for col_idx, fname in enumerate(filenames):
            ax       = axes[row_idx, col_idx]
            img_path = os.path.join(IMAGES_DIR, concept, t_bin, fname)
            try:
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
            except FileNotFoundError:
                ax.text(0.5, 0.5, "missing", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8, color="red")
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out = os.path.join(RESULTS_DIR, f"grid_{concept}_{lang}_{m}.png")
    plt.savefig(out, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")


# ── Curated paper-quality grids ────────────────────────────────────────────

PAPER_GRIDS = [
    # (concept,      lang,  model,      rationale)
    ("television",  "en",  "clip",     "Semantic Fossilization — EN archetype lock"),
    ("television",  "bn",  "clip",     "Fossilization contrast — BN no archetype"),
    ("phone",       "en",  "clip",     "Technology evolution — CLIP/M-CLIP"),
    ("phone",       "ja",  "clip",     "Morphological discontinuity — JA lexical jump"),
    ("protest",     "en",  "clip",     "Anomaly — near-random trajectory"),
    ("car",         "en",  "clip",     "Fast-drift validation — automotive design"),
    ("surgery",     "en",  "clip",     "Slow-drift validation — OR modernisation"),
    ("mountain",    "en",  "clip",     "Stable baseline + photographic artifact"),
    ("television",  "en",  "siglip2",  "Fossilization — SigLIP cross-architecture check"),
    ("phone",       "en",  "siglip2",  "Technology evolution — SigLIP"),
]

print("Generating curated paper-quality retrieval grids...\n")
print(f"{'CONCEPT':<14} {'LANG':>5} {'MODEL':>10}  RATIONALE")
print("-" * 70)
for concept, lang, model_key, reason in PAPER_GRIDS:
    print(f"  {concept:<12} {lang:>5} {model_key:>10}  {reason}")
print()

for concept, lang, model_key, reason in PAPER_GRIDS:
    if concept not in CONFIG["concepts"]:
        print(f"Skipping {concept} (not in current CONFIG)")
        continue
    print(f"\n── {concept.upper()} | {lang.upper()} | {model_key.upper()} ──")
    print(f"   {reason}")
    build_retrieval_grid(concept, lang, k=5, model=model_key)
